# Policy network: DDPG allocation of bandwidth and power to several 360° videos

**5 videos · 50 headsets choosing by Zipf · 5-s intervals · one step per watched video · reward: Seer's QoE**

This notebook is the third block of the pipeline. The first two blocks are used here **frozen**: their weights are
loaded and never changed. They are **one FoV predictor $f_\theta^{(v)}$ per video** and **one CVAE-GAN encoder
common to all videos**:

$$F^{(v)}_{k-5},\dots,F^{(v)}_{k-1}\ \xrightarrow{\ f^{(v)}_\theta\ }\ \hat F^{(v)}_k\ \xrightarrow{\ (\hat F-\bar X)/s\ }\ \xrightarrow{\ \text{encoder}\ }\ z^{(v)}_k\ \xrightarrow{\ \text{sigmoid}\ }\ \textbf{policy}\ \mu(S^{(v)}_k)\ \longrightarrow\ a^{(v)}_k = [\,b_0,\ b_1..b_L,\ \rho\,]$$

> **Files needed first** (section 3), in `outputs_predictor_vae_combined/`:
> - `predictor_<video>.pt` for each of the 5 videos, from `Predictor_VAE_Combined.ipynb` run once per video;
> - one `cvaegan_encoder.pt`, trained in `cvae_gan_normalised_and_diff_weights.ipynb` on the pooled predicted
>   frames of all 5 videos.

### The set-up in one table

| item | what it is here |
|---|---|
| **videos** | $V = 5$ (`coaster`, `coaster2`, `diving`, `drive`, `game`), streamed at the same time over one cell |
| **headsets** | 50. Each picks a video by a **Zipf law** with $s = 0.8$: $p_v \propto v^{-0.8}$, i.e. 0.39 / 0.22 / 0.16 / 0.13 / 0.11. Almost always all 5 are watched |
| **viewing behaviours** | per video, a crowd of **10** dataset viewers. Their average is the predictor's input, and each headset on that video follows one of the 10 behaviours as its true FoV |
| **interval** $k$ | one 5-s segment, $k = 6,\dots,12$: **7 intervals per episode** |
| **step** | one watched video in one interval, served in rank order. An episode has $7 \times V_\text{active}$ steps (35 when all 5 are watched) |
| **state** $S^{(v)}_k$ | $[\mathrm{sigmoid}(z), w, \mu, \sigma, \bar q, q^{\min}, B_\text{rem}/B_\text{tot}, P_\text{rem}/P_\text{tot}, N_v/50, \text{videos left}/V]$: **28 numbers** |
| **action** $a^{(v)}_k$ | $b_0$ = share of the free band **kept** for later videos, $b_1..b_L$ = shares of the free band for the 10 attention groups (softmax, so $b_0 + \dots + b_L = 1$), $\rho$ = share of the free **power** (sigmoid): **12 numbers** |
| **layer bitrates** | R-Q model anchored at **40 Mbit/s for all layers of a 4K frame**, scaled by pixel count, so 160 Mbit/s at 8K |
| **reward** $r^{(v)}_k$ | **Seer's QoE** (Kumar et al., IEEE TNSE 2023, Eq. 6), averaged over the video's headsets and weighted by $N_v/50$: $R = \lambda Q - \mu T - \xi\lvert\Delta Q\rvert$ |
| **scenarios** | {4K, 8K} × {20 MHz, 10 MHz}, one agent each |
| **agent** | DDPG with one common replay buffer for the steps of all videos. $\gamma \in \{0.9, 0.95, 0.99\}$ is swept on 8K-20MHz, and the best $\gamma$ is reused for the other three scenarios |

### What changed in this version
1. **Several videos at once** (section 7). One step per watched video, sequentially; the band and the power are
   shared. Each step's state shows how much bandwidth and power are still free.
2. **Transmit power is part of the action** ($\rho$). The SNR of a video's headsets depends on its power and its band.
3. **50 headsets, Zipf popularity**, 10 viewing behaviours per video.
4. **One predictor per video, one common encoder** (section 3).
5. **Seer's reward** (section 7): quality minus rebuffering minus quality changes. The earlier utilities $U_b$, $U_d$
   are still computed and reported, but are no longer in the reward.
6. **Bitrate ladder anchored at 40 Mbit/s for 4K**, and **four scenarios** (4K/8K × 20/10 MHz).

## Read this first: the whole policy network in plain words

### 1. What decision is being made?
Every 5 seconds the edge server must send the next 5-s piece of **five** 360° videos to 50 headsets over one radio
cell. Every headset watches one video; a popular video has many viewers, an unpopular one few or none. A video is
**multicast**: each tile is sent once, and every headset watching it receives it at the rate its own channel
allows. Each of the 200 tiles is coded in **SVC layers**: a base layer plus up to 9 enhancement layers.

The limited resources are the **bandwidth** (20 or 10 MHz) and the **transmit power** (23 dBm). In every
interval the policy is asked once per watched video, in order of popularity:
* how much of the bandwidth that is still free this video takes (the rest is kept for the videos after it);
* how to split that bandwidth over the video's 10 attention groups;
* how much of the free power it takes.

The last video of the interval gets whatever is left.

### 2. Which tiles get how many layers?
Unchanged. The video's predictor says how much of its crowd will look at each tile ($\hat F_k$). The tiles are
sorted into 10 groups by that number, and a tile in group $l$ is meant to receive layers $1..l$.

### 3. What does the policy see?
For the video being served:
* the 10-number summary $z$ of its predicted frame;
* the mean attention of each group;
* the average and spread of its viewers' radio path loss;
* their playout buffers;
* how much bandwidth and power are still free;
* its share of the 50 headsets;
* how many watched videos still come after it.

That is 28 numbers.

### 4. How is a decision scored? (Seer's QoE)
For every headset: the **quality** $Q$ it receives, minus its **rebuffering** time $T$ (seconds of frozen
playback), minus how much the quality **changed** since the last interval. A video's reward is the mean over its
viewers, times its share of the 50 headsets. So one interval's rewards add up to the mean QoE over all 50 headsets,
and popular videos count more.

### 5. How does it learn? (DDPG)
As before: a critic learns how much reward follows a state and an action, and the actor learns the actions the
critic rates highest. The experience comes from a **replay buffer**. All videos' steps go into the same buffer,
and one actor serves them all.

### 6. How do we know it is any good?
It is compared, on the same test episodes, with 8 rules that need no learning. Each rule combines a split over the
groups (attention, equal per tile, demand, random) with a split over the videos (equal, or by audience). Power
follows bandwidth in every rule.

### 7. Map of this notebook

| section | what happens |
|---|---|
| 1 | hardware, folders |
| 2 | every setting in one place: videos, Zipf, channel, SVC ladder, Seer's reward, scenarios |
| 3 | the frozen predictors (one per video) and the common encoder are loaded |
| 4 | training, validation and test crowds of every video, with their predicted frames |
| 5 | the quantile masks $M_l$, $T_l$, $w_l$ |
| 6 | the radio channel and the path-loss statistics $\mu$, $\sigma$ |
| 7 | the environment: Zipf choices, sequential steps, bandwidth and power, playout queue, reward |
| 8 | actor and critic |
| **9** | **DDPG training: the $\gamma$ sweep on 8K-20MHz, then the other three scenarios** |
| 10 | training graphs and the critic's Q |
| 11 | test: DDPG against the 8 rules, in every scenario |
| 12 | one interval up close |
| 13 | files saved, points to confirm |

## 1. Setup and hardware

The same folder search, output folder and device choice as `Predictor_VAE_Combined.ipynb` (sections 0 and 1
there), so both notebooks read the same data and write to the same `outputs_predictor_vae_combined/` folder.

In [ ]:
import os, re, glob, time, json, random, platform, shutil
from dataclasses import dataclass, asdict
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
from torch import nn
import matplotlib.pyplot as plt
%matplotlib inline

print("PyTorch:", torch.__version__)

def find_data_root():
    for base in ("/kaggle/input", ".", "../dataset", ".."):
        if not os.path.isdir(base):
            continue
        for root, dirs, _ in os.walk(base):
            if os.path.isdir(os.path.join(root, "sensory", "tile")):
                return root
    raise FileNotFoundError("could not find a folder containing sensory/tile")

DATA_ROOT = find_data_root()
WORK_DIR  = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
OUT_DIR   = os.path.join(WORK_DIR, "outputs_predictor_vae_combined")
os.makedirs(OUT_DIR, exist_ok=True)
print("DATA_ROOT =", os.path.abspath(DATA_ROOT))
print("outputs   ->", os.path.abspath(OUT_DIR))

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.width", 200)


def pick_device():
    # Use a GPU when one is present and works, otherwise the CPU.
    found = []
    if torch.cuda.is_available():                                          # NVIDIA, or AMD via ROCm
        found.append(("cuda", torch.cuda.get_device_name(0)))
    if hasattr(torch, "xpu") and torch.xpu.is_available():                 # Intel Arc
        found.append(("xpu", torch.xpu.get_device_name(0)))
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():   # Apple Silicon
        found.append(("mps", "Apple Silicon GPU"))

    for kind, name in found:
        dev = torch.device(kind)
        try:   # a PyTorch build can list a GPU it has no kernels for, so run a real conv on it
            x = torch.ones(1, 1, 4, 4, device=dev)
            nn.functional.conv2d(x, torch.ones(1, 1, 3, 3, device=dev)).sum().item()
            return dev, name
        except Exception as err:
            print(f"[warn] {kind} device '{name}' was found but cannot run: {err}")
    return torch.device("cpu"), platform.processor() or platform.machine()


DEVICE, DEVICE_NAME = torch.device("cpu"), "cpu"

# full float32 on NVIDIA too (no TF32), and deterministic cuDNN kernels
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

print(f"device: {DEVICE.type}  ({DEVICE_NAME})")

## 2. Settings

Every number the policy network depends on lives in the one code cell below. Values marked **(assumed)** should be
confirmed.

### Videos and viewers

| setting | value | meaning |
|---|---|---|
| `videos` | coaster, coaster2, diving, drive, game | the $V = 5$ videos, **in Zipf rank order** (assumed: the dataset has no popularity data) |
| `zipf_s` | 0.8 | popularity $p_v = v^{-s} / \sum_{v'} v'^{-s}$: 0.39, 0.22, 0.16, 0.13, 0.11 |
| `n_users` | 50 | headsets in the cell; each picks one video at the start of an episode and keeps it |
| `crowd_size` | 10 | viewing behaviours per video: the crowd whose average is the predictor's input, and whose 10 behaviours the video's headsets follow |

With 50 headsets the least popular video goes unwatched only about 0.4% of the time (4.99 watched videos
per episode on average). An unwatched video gets no step and no resources.

### Radio channel (`main.pdf`, Table 1, after Liang et al. [1])

| setting | value | meaning |
|---|---|---|
| `f_c_hz` | 3.5 GHz | carrier frequency $f_c$ |
| `B_tot` | 20 or 10 MHz (scenario) | total downlink bandwidth, **shared by all videos** |
| `P_t_dBm` | 23 dBm | **total** edge server transmit power, shared by all videos |
| `G_t_dBi`, `G_r_dBi` | 8 dBi, 3 dBi | antenna gains |
| `N0_dBm_hz`, `nf_db` | −174 dBm/Hz, 9 dB | noise PSD $N_0$ and noise figure $F$; the noise over a band $B$ is $N_0 F B$ |
| `radius_m`, `d_min_m` | 200 m, 35 m | headsets placed uniformly over the ring $35 \le d \le 200$ m |
| `d0_m` | 1 m | reference distance of the path loss |
| `shadow_db`, `shadow_dcorr_m` | 8 dB, 50 m | log-normal shadowing and its decorrelation distance |
| `walk_mps` | 1.5 m/s | headset walking speed |

### SVC layer bitrates: R-Q model anchored at 40 Mbit/s for 4K

The bitrate a tile needs grows as the encoder's quantisation parameter QP falls. HEVC's quantiser step doubles every
6 QP, so the bitrate roughly doubles for every 6 QP less (the "rule of 6"): $R^\text{single}(QP) \propto
e^{-\beta QP}$, $\beta = \ln 2 / 6$. Layer $k = 1..10$ is encoded at $QP_k = 38 - 2(k-1)$, and SVC coding costs an
overhead $\alpha_k = 0.09(k-1)$. The **shape** of the cumulative ladder is therefore

$$R_k \propto (1 + \alpha_k)\,e^{-\beta (QP_k - 38)},\qquad \frac{R_{10}}{R_1} = 1.81 \cdot 2^3 = 14.48 .$$

The **level** is set by one anchor: all 10 layers on all 200 tiles of a **4K** (3840 × 1920) frame = **40 Mbit/s**.
That is the middle of YouTube's recommended 35-45 Mbps for 4K at 24-30 fps
(support.google.com/youtube/answer/1722171). So $R_{10} = 200$ kbit/s per 192 × 192-pixel tile, and the 4K ladder
(cumulative, kbit/s per tile) is

13.8 · 19.0 · 25.9 · 35.1 · 47.3 · 63.6 · 85.1 · 113.5 · 150.9 · 200.0.

The rate **per pixel** is kept, so at **8K** (7680 × 3840, 384 × 384-pixel tiles) every rate is 4× larger: 55 …
800 kbit/s per tile, **160 Mbit/s** for the whole frame. That is the top of YouTube's 80-160 Mbps for 8K, which serves
as a consistency check. Two caveats:
- These are upload (mezzanine) rates; delivery ladders in the literature top out lower (about 20-23.5 Mbps for 4K,
  arXiv 1711.03362).
- The dataset videos are not 8K. The 8K scenarios assume **a hypothetical 8K encoding of the same content**, with
  head motion that does not depend on resolution.

| setting | value | meaning |
|---|---|---|
| `T_seg` | 5 s | one interval = one segment |
| `video_px` | (7680, 3840) | resolution of the base scenario; the 4K scenarios use (3840, 1920) |
| `top_frame_bps`, `top_frame_px` | 40 Mbit/s, 3840 × 1920 | the anchor: all layers of a 4K frame |
| `qp_base`, `qp_step` | 38, 2 | QP of the base layer and the step per enhancement layer (assumed) |
| `rq_beta` | ln 2 / 6 | rule of 6 |
| `svc_overhead` | 0.09 | $\alpha_k = 0.09(k-1)$ |

### Playout queue (assumed)

| setting | value | meaning |
|---|---|---|
| `chunk_s` | 1 s | a segment is played as 5 chunks; freezes are computed per chunk |
| `D0_s` | 0.5 s | playout delay at the start |
| `carry_queue` | True | a freeze carries over to later segments |

### Reward: Seer's QoE

Kumar et al., "Seer: Learning-Based 360° Video Streaming for MEC-Equipped Cellular Networks", IEEE TNSE 10(6),
2023, Eq. (6), with the weights of its Sec. V:

$$\text{QoE} = \lambda\sum_k Q(q_k) - \mu\sum_k T_k - \xi\sum_{k\ge2}\big|Q(q_k) - Q(q_{k-1})\big|,\qquad \lambda = 1,\ \mu = 4.3,\ \xi = 1 .$$

In Seer, $Q$ is linear in the bitrate (0.30 … 4.3 for 360p … 4K), so $\mu = 4.3$ is its **top quality**: one second of
rebuffering costs as much as a segment at top quality. Section 7 maps $q$ and $T$ onto this notebook.

| setting | value | meaning |
|---|---|---|
| `q_mode` | "normalised" | **open choice.** "normalised" (option A): $Q = q / Q_\text{max} \in [0,1]$. "mbps" (option B): $Q = q$ in Mbit/s |
| `lam_q`, `xi_smooth` | 1, 1 | $\lambda$, $\xi$ of Seer |
| `mu_rebuf` | None | $\mu$; `None` = $Q_\text{max}$ in the units of $Q$, as in Seer: 1 with option A, 160 (8K) or 40 (4K) with option B |

### Scenarios

| scenario | video | band |
|---|---|---|
| **8K-20MHz** (base: the $\gamma$ sweep runs here) | 7680 × 3840 | 20 MHz |
| 8K-10MHz | 7680 × 3840 | 10 MHz |
| 4K-20MHz | 3840 × 1920 | 20 MHz |
| 4K-10MHz | 3840 × 1920 | 10 MHz |

### Path-loss statistics, state scaling, DDPG

| setting | value | meaning |
|---|---|---|
| `ewma_a`, `ewma_b` | 0.3, 0.3 | the moving averages of $\mu$ and $\sigma$ (per video, over its headsets) |
| `pl_ref_db`, `pl_scale_db` | 100, 10 | the state holds $(\mu - 100)/10$ and $\sigma/10$ |
| `hidden` | 128 | width of the hidden layers |
| `GAMMAS` | 0.9, 0.95, 0.99 | swept on 8K-20MHz; the best is reused for the other scenarios |
| `lr_actor`, `lr_critic` | $10^{-4}$, $10^{-3}$ | Adam learning rates |
| `soft_tau` | 0.01 | $\kappa$ of the soft target updates |
| `buffer`, `batch`, `min_buffer` | 50,000, 256, 50 | one common replay memory; $M$ samples per update |
| `updates_per_step` | 2 | gradient steps after every step (every video, every interval) |
| `eps0`, `eps_decay`, `noise_std` | 1, 0.99, 1 | exploration |
| `episodes` | 6000 | per training: about 35 steps each, so about 210,000 steps and 420,000 updates |
| `n_crowds`, `val_frac`, `n_val_episodes` | 200 per video, 0.2, 40 | crowds per video; validation episodes |

**Why $\gamma > 0$.** A freeze grows the viewer's buffer for later intervals, and bandwidth or power one video takes
is missing for the videos after it in the same interval. Both effects reach later steps, so the future counts.

In [ ]:
from dataclasses import replace


@dataclass
class PCfg:
    # ---- videos and viewers ----
    videos: tuple = ("coaster", "coaster2", "diving", "drive", "game")   # Zipf rank 1..V, in this order (assumed)
    zipf_s: float = 0.8             # popularity: p_v proportional to rank^-s
    n_users: int = 50               # headsets in the cell; each picks one video
    crowd_size: int = 10            # viewing behaviours per video (the crowd behind the predictor's input)

    # ---- radio channel (main.pdf, Table 1; model of Liang et al. [1]) ----
    f_c_hz: float = 3.5e9           # carrier frequency
    B_tot: float = 20e6             # total downlink bandwidth, Hz, shared by all videos
    P_t_dBm: float = 23.0           # total edge server transmit power P, shared by all videos
    G_t_dBi: float = 8.0            # edge server antenna gain
    G_r_dBi: float = 3.0            # headset antenna gain
    N0_dBm_hz: float = -174.0       # thermal noise PSD
    nf_db: float = 9.0              # headset receiver noise figure F
    radius_m: float = 200.0
    d_min_m: float = 35.0           # the path-loss model holds for d >= 35 m
    d0_m: float = 1.0               # reference distance d_0
    shadow_db: float = 8.0          # xi, std of the log-normal shadowing X_{u,k}
    shadow_dcorr_m: float = 50.0    # shadowing decorrelation distance
    walk_mps: float = 1.5           # headset walking speed (0 = static users, fixed shadowing)

    # ---- EWMA of the path loss (methodology alpha, beta) ----
    ewma_a: float = 0.3
    ewma_b: float = 0.3

    # ---- SVC layers: R-Q model anchored at 40 Mbit/s for a 4K frame ----
    T_seg: float = 5.0
    video_px: tuple = (7680, 3840)  # base scenario: 8K (hypothetical re-encoding); SCENARIOS also runs 4K
    top_frame_bps: float = 40e6     # all L layers on all tiles of a frame of ...
    top_frame_px: int = 3840 * 1920 # ... this many pixels (YouTube: 35-45 Mbps for 4K at 24-30 fps)
    qp_base: float = 38.0           # QP of layer 1
    qp_step: float = 2.0            # QP decrease per enhancement layer: 38, 36, ..., 20
    rq_beta: float = float(np.log(2) / 6)   # R(QP) = alpha e^(-beta QP); rule of 6: rate x2 per -6 QP
    svc_overhead: float = 0.09      # alpha_k = svc_overhead * (k - 1)

    # ---- playout queue  <-- ASSUMED, confirm ----
    chunk_s: float = 1.0            # the freeze is computed per 1-s chunk
    D0_s: float = 0.5               # playout delay of the first chunk at the start of the video
    carry_queue: bool = True        # a freeze carries over to later segments (False: reset to D0_s each segment)

    # ---- reward: Seer's QoE (Kumar et al., IEEE TNSE 2023, Eq. 6)  R = lam Q - mu T - xi |Q - Q_prev| ----
    q_mode: str = "normalised"      # "normalised": Q = q / Q_max (option A);  "mbps": Q = q in Mbit/s (option B)
    lam_q: float = 1.0              # lambda
    mu_rebuf: object = None         # mu; None = Q_max in the units of Q, as in Seer (mu = 4.3 = its top Q)
    xi_smooth: float = 1.0          # xi

    # ---- state scaling ----
    pl_ref_db: float = 100.0
    pl_scale_db: float = 10.0

    # ---- crowds ----
    n_crowds: int = 200             # random 10-viewer crowds per video
    val_frac: float = 0.2
    n_val_episodes: int = 40        # validation episodes (fixed seeds)

    # ---- DDPG (the discount gamma is set per agent, see GAMMAS) ----
    hidden: int = 128
    lr_actor: float = 1e-4
    lr_critic: float = 1e-3
    soft_tau: float = 0.01          # kappa in theta' <- (1 - kappa) theta' + kappa theta
    buffer: int = 50000             # one common memory for the steps of all videos
    batch: int = 256                # M
    min_buffer: int = 50            # train once the memory holds more than this
    updates_per_step: int = 2       # gradient steps (step 2 of the notes) after every step
    eps0: float = 1.0
    eps_decay: float = 0.99
    noise_std: float = 1.0          # N(0, 1) on the actor's logits
    episodes: int = 6000            # per training
    eval_every: int = 25
    n_test_seeds: int = 50          # test episodes per policy and scenario
    seed: int = 0

    @property
    def P_t_W(self):     return 10 ** ((self.P_t_dBm - 30) / 10)
    @property
    def gain_db(self):   return self.G_t_dBi + self.G_r_dBi
    @property
    def noise_W(self):   return self.noise_at(self.B_tot)                                   # N0 F B_tot

    def noise_at(self, B_hz):
        return 10 ** ((self.N0_dBm_hz + self.nf_db - 30) / 10) * B_hz                          # N0 F B


pcfg = PCfg()
GAMMAS = (0.9, 0.95, 0.99)          # swept on the base scenario; the best is reused for the others

# ---- scenarios: video resolution x total bandwidth; one agent each ----
SCENARIOS = {"8K-20MHz": dict(video_px=(7680, 3840), B_tot=20e6),
             "8K-10MHz": dict(video_px=(7680, 3840), B_tot=10e6),
             "4K-20MHz": dict(video_px=(3840, 1920), B_tot=20e6),
             "4K-10MHz": dict(video_px=(3840, 1920), B_tot=10e6)}
BASE_SCEN = "8K-20MHz"              # the gamma sweep runs here
SCEN_CFG = {name: replace(pcfg, **kw) for name, kw in SCENARIOS.items()}
assert SCEN_CFG[BASE_SCEN] == pcfg

# ---- Zipf popularity of the videos ----
V_N = len(pcfg.videos)
P_ZIPF = np.arange(1, V_N + 1, dtype=float) ** -pcfg.zipf_s
P_ZIPF /= P_ZIPF.sum()
EXP_ACTIVE = float((1 - (1 - P_ZIPF) ** pcfg.n_users).sum())    # expected number of videos watched


def cumulative_rates(L, tile_px, p=pcfg):
    # R_k in bit/s per tile: decode layers 1..k. Shape of the R-Q model: QP_k = qp_base - qp_step (k-1),
    # single-layer rate ~ e^(-beta QP_k), SVC overhead (1 + alpha_k); scaled so that R_L = top_frame_bps per
    # top_frame_px pixels (the rate per pixel is the same at every resolution). Returns (R_1..R_L, QP_1..QP_L).
    k = np.arange(1, L + 1)
    qp = p.qp_base - p.qp_step * (k - 1)
    shape = (1 + p.svc_overhead * (k - 1)) * np.exp(-p.rq_beta * (qp - p.qp_base))
    return p.top_frame_bps * (tile_px / p.top_frame_px) * shape / shape[-1], qp


def layer_rates(L, tile_px, p=pcfg):
    # r_k = R_k - R_{k-1}: the bitrate of layer k on its own, bit/s per tile
    return np.diff(cumulative_rates(L, tile_px, p)[0], prepend=0.0)


print(f"videos     {', '.join(f'{v} ({p_:.2f})' for v, p_ in zip(pcfg.videos, P_ZIPF))}  <- Zipf s = {pcfg.zipf_s}")
print(f"viewers    {pcfg.n_users} headsets; {EXP_ACTIVE:.2f} of {V_N} videos watched on average; "
      f"{pcfg.crowd_size} viewing behaviours per video")
print(f"power      P = {pcfg.P_t_dBm} dBm = {pcfg.P_t_W:.2f} W in total,  G_t + G_r = {pcfg.gain_db:.0f} dBi")
print(f"SVC        top {pcfg.top_frame_bps/1e6:g} Mbit/s for all layers of a {pcfg.top_frame_px:,}-px (4K) frame, "
      f"QP {pcfg.qp_base:g} - {pcfg.qp_step:g}(k-1), beta = ln2/6, overhead {pcfg.svc_overhead} (k-1)")
for name, c in SCEN_CFG.items():
    print(f"scenario   {name:9s} video {c.video_px[0]} x {c.video_px[1]} px, B_tot = {c.B_tot/1e6:g} MHz, "
          f"noise N0 F B_tot = {10 * np.log10(c.noise_W) + 30:.1f} dBm" + ("   <- gamma sweep" if name == BASE_SCEN else ""))
print(f"playout    {pcfg.chunk_s:g}-s chunks, D0 = {pcfg.D0_s} s, carry_queue = {pcfg.carry_queue}")
print(f"reward     Seer: R = {pcfg.lam_q:g} Q - mu T - {pcfg.xi_smooth:g} |dQ|, Q mode '{pcfg.q_mode}', "
      f"mu = {'Q_max (Seer)' if pcfg.mu_rebuf is None else pcfg.mu_rebuf}")
print(f"DDPG       gamma in {GAMMAS} on {BASE_SCEN}, {pcfg.episodes} episodes per training, lr actor {pcfg.lr_actor} / "
      f"critic {pcfg.lr_critic}, kappa {pcfg.soft_tau}, M = {pcfg.batch} x {pcfg.updates_per_step}/step, "
      f"eps {pcfg.eps0} x {pcfg.eps_decay}/episode")

## 3. The frozen front end: one predictor per video, one common encoder

| file | contents | written by |
|---|---|---|
| `predictor_weights_<video>.pt` (one per video) | the predictor of that video: weights $\theta^{(v)}$ and its settings `cfg` (video, grid, order $n$, fps) | `Predictor_VAE_Combined.ipynb`, run with `cfg.video = <video>` |
| `cvaegan_encoder.pt` | encoder weights, $L$, hidden sizes, $\tau$, the normalisation statistics $\bar X$, $s$, and (expected) the list `videos` it was trained on | `cvae_gan_normalised_and_diff_weights.ipynb`, trained on the **pooled** predicted frames of all 5 videos |

The cell stops with a clear message if any file is missing. It checks that each predictor belongs to its video and
that all of them use the same grid, order and sampling. It warns if the encoder does not list the 5 videos: the
earlier encoder was trained on coaster frames only, so the other videos' frames would lie outside its training
data.

Every predicted frame is normalised with the saved $\bar X$ and $s$ before it is encoded, and $z$ is drawn afresh
every interval:

$$\hat F^{(v)}_k \xrightarrow{\ (\hat F - \bar X)/s\ } \xrightarrow{\text{encoder}} (\mu_z, \log\sigma^2_z) \xrightarrow{\ z \sim \mathcal N(\mu_z, \sigma^2_z)\ } \mathrm{sigmoid}(z) \in (0,1)^{L}.$$

**Test crowd per video.** If a predictor file holds a held-out `U_test`, that set is used; otherwise it is
`U[::5]` (10 viewers, fold 1 of the CVAE notebook). The front end may have seen these viewers; the **policy** never
trains on them.

**`PRED_OVERRIDE`** (empty) lets a smoke test run before the per-video predictors exist, by pointing videos at
another video's predictor. It prints a warning, and its results are not valid.

In [ ]:
ENC_FILE = os.path.join(OUT_DIR, "cvaegan_encoder.pt")
PRED_OVERRIDE = {}      # {video: path}. ONLY for a smoke test before the per-video predictors exist; never for results
PRED_FILE = {v: PRED_OVERRIDE.get(v, os.path.join(OUT_DIR, f"predictor_weights_{v}.pt")) for v in pcfg.videos}
missing = [f for f in (ENC_FILE, *PRED_FILE.values()) if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(
        "missing:\n  " + "\n  ".join(missing) + "\n"
        "Run Predictor_VAE_Combined.ipynb once per video (cfg.video = <video>) and save each predictor as "
        "predictor_weights_<video>.pt; train ONE cvaegan_encoder.pt in cvae_gan_normalised_and_diff_weights.ipynb on the "
        f"pooled predicted frames of all videos {pcfg.videos}. Copy them all to {OUT_DIR}.")
if PRED_OVERRIDE:
    print(f"[WARN] PRED_OVERRIDE is set for {sorted(PRED_OVERRIDE)}: those videos use another video's predictor. "
          f"Use this only for a smoke test; the results are not valid.")

# weights_only=False: the files also hold settings and numpy arrays (the normalisation statistics)
ENC_CK  = torch.load(ENC_FILE, map_location="cpu", weights_only=False)
PRED_CK = {v: torch.load(PRED_FILE[v], map_location="cpu", weights_only=False) for v in pcfg.videos}
if "videos" not in ENC_CK:
    print("[WARN] cvaegan_encoder.pt does not list the videos it was trained on: if it is the single-video "
          "(coaster) encoder, the other videos' frames are outside its training data.")
elif set(ENC_CK["videos"]) != set(pcfg.videos):
    print(f"[WARN] the encoder was trained on {ENC_CK['videos']}, the policy streams {pcfg.videos}")

CFG = {}
for v in pcfg.videos:                               # each predictor's settings (video, grid, order n, fps, ...)
    c = SimpleNamespace(**PRED_CK[v]["cfg"])
    c.n_samples = c.video_seconds // c.sample_every_s
    if v not in PRED_OVERRIDE:
        assert c.video == v, f"{PRED_FILE[v]} holds the predictor of '{c.video}', not of '{v}'"
    CFG[v] = c
cfg = CFG[pcfg.videos[0]]                           # the grid, order n and sampling every video shares
for v, c in CFG.items():
    assert (c.H, c.W, c.n, c.n_samples) == (cfg.H, cfg.W, cfg.n, cfg.n_samples), f"'{v}' uses other settings"
L, ENC_HIDDEN, ENC_LEAK = int(ENC_CK["L"]), tuple(ENC_CK["hidden"]), float(ENC_CK["leak"])
assert tuple(ENC_CK["input_shape"]) == (cfg.H, cfg.W), "the encoder and the predictors use different grids"
assert ENC_CK.get("latent_sigmoid", True), "the policy expects sigmoid(z_k)"
NORMALISED = bool(ENC_CK.get("normalised", False))
NORM_MEAN  = np.asarray(ENC_CK["norm_mean"], np.float32) if NORMALISED else None    # X_bar, (H, W)
NORM_SCALE = float(ENC_CK.get("norm_scale", 1.0))                                     # s


# ---- the same functions as Predictor_VAE_Combined.ipynb and cvae_gan_normalised_and_diff_weights.ipynb ----
def parse_tile_csv(path, W=cfg.W, H=cfg.H):
    # Ragged 'frame, tile, tile, ...' CSV -> (n_frames, H, W) 0/1 array.
    occ = []
    with open(path) as f:
        f.readline()
        for line in f:
            parts = [p.strip() for p in line.strip().split(",")]
            if not parts or parts[0] == "":
                continue
            g = np.zeros((H, W), np.float32)
            for x in (int(p) for p in parts[1:] if p != ""):
                j, i = divmod(x, W)                 # row j (latitude), column i (longitude)
                if 0 <= j < H and 0 <= i < W:
                    g[j, i] = 1.0
            occ.append(g)
    return np.stack(occ)


def fov_samples(occ, c=cfg):
    # The 12 samples G_{u,1}..G_{u,12}: the frame at 5, 10, ..., 60 s.
    step = c.sample_every_s * c.fps
    idx = [min(k * step, len(occ)) - 1 for k in range(1, c.n_samples + 1)]
    return occ[idx], idx


def combined(Gv, users, c=cfg):
    # F_k = (1/|users|) sum_{u in users} G_{u,k},  for k = 1..12  (Gv: the maps of one video)
    return {k: np.mean([Gv[u][k - 1] for u in users], 0)
            for k in range(1, c.n_samples + 1)}


def combined_pairs(Fk, c=cfg):
    # X = [F_{k-n}, ..., F_{k-1}] (oldest first), Y = F_k,  for k = n+1 .. 12
    ks = list(range(c.n + 1, c.n_samples + 1))
    X = np.stack([np.moveaxis(np.stack([Fk[i] for i in range(k - c.n, k)]), 0, -1)
                  for k in ks]).astype(np.float32)
    Y = np.stack([Fk[k] for k in ks]).astype(np.float32)
    return X, Y, ks


class CircularPad(nn.Module):
    # Wrap columns (longitude), zero-pad rows (latitude). x is (batch, C, H, W).
    def __init__(self, p):
        super().__init__(); self.p = p
    def forward(self, x):
        p = self.p
        x = torch.cat([x[..., -p:], x, x[..., :p]], dim=3)
        return nn.functional.pad(x, (0, 0, p, p))


class Order5Predictor(nn.Module):
    def __init__(self, c=cfg):
        super().__init__()
        self.pad1  = CircularPad(1)
        self.conv1 = nn.Conv2d(c.n, c.width, 3)
        self.pad2  = CircularPad(2)
        self.conv2 = nn.Conv2d(c.width, c.width, 3, dilation=2)
        self.head  = nn.Conv2d(c.width, 1, 1)

    def forward(self, x):
        h = x.permute(0, 3, 1, 2)
        h = torch.relu(self.conv1(self.pad1(h)))
        h = torch.relu(self.conv2(self.pad2(h)))
        return torch.sigmoid(self.head(h)).squeeze(1)          # G_hat_{u,k}: (batch, H, W)


def mlp(sizes, leak=ENC_LEAK):
    # Linear -> LeakyReLU for every consecutive pair of widths
    return [m for a, b in zip(sizes, sizes[1:]) for m in (nn.Linear(a, b), nn.LeakyReLU(leak))]


class Encoder(nn.Module):
    # X' = normalised F_hat_k  ->  (mu_{z,k}, log sigma_{z,k}^2)
    def __init__(self, L=L, hidden=ENC_HIDDEN, leak=ENC_LEAK, c=cfg):
        super().__init__()
        sizes = [c.H * c.W, *hidden]                            # 200 -> 128 -> 64
        self.body  = nn.Sequential(*mlp(sizes, leak))
        self.to_mu = nn.Linear(sizes[-1], L)                   # mu_{z,k}
        self.to_lv = nn.Linear(sizes[-1], L)                   # log sigma_{z,k}^2

    def forward(self, X):
        h = self.body(X.flatten(1))
        return self.to_mu(h), self.to_lv(h)


def draw_z(mu, lv):
    # one random value per latent slot from N(mu_{z,k}, sigma_{z,k}^2)
    return mu + torch.exp(0.5 * lv) * torch.randn_like(mu)


# ---- the frozen weights: one predictor per video, one common encoder ----
PRED = {}
for v in pcfg.videos:
    PRED[v] = Order5Predictor(CFG[v]).to(DEVICE)
    PRED[v].load_state_dict(PRED_CK[v]["predictor"]); PRED[v].eval()
enc = Encoder().to(DEVICE)
enc.load_state_dict(ENC_CK["encoder"]); enc.eval()
for p_ in (*(q for m in PRED.values() for q in m.parameters()), *enc.parameters()):
    p_.requires_grad_(False)


@torch.no_grad()
def predict(v, X_t):
    # the frozen predictor of video v
    return PRED[v](X_t).cpu().numpy()


def encoder_input(F_hat):
    # the encoder was trained on X' = (X - X_bar) / s, with the X_bar and s saved in cvaegan_encoder.pt
    F = np.asarray(F_hat, np.float32).reshape(-1, cfg.H, cfg.W)
    return ((F - NORM_MEAN) / NORM_SCALE).astype(np.float32) if NORMALISED else F


@torch.no_grad()
def z_for_policy(F_hat_k, net=enc):
    # serving path: F_hat_k -> normalise -> one draw z_k ~ N(mu_{z,k}, sigma_{z,k}^2) -> sigmoid(z_k) in (0,1)^L
    net.eval()
    mu, lv = net(torch.from_numpy(encoder_input(F_hat_k)).to(DEVICE))
    return torch.sigmoid(draw_z(mu, lv)).cpu().numpy()


# ---- every video's viewers and their tile maps G_{u,1..12} ----
tile_dir = os.path.join(DATA_ROOT, "sensory", "tile")
G, U, U_test = {}, {}, {}
for v in pcfg.videos:
    rx = re.compile(rf"^{re.escape(v)}_(user\d+)_tile\.csv$")
    U[v] = sorted(m.group(1) for p in glob.glob(os.path.join(tile_dir, "*_tile.csv"))
                  if (m := rx.match(os.path.basename(p))))
    assert len(U[v]) > pcfg.crowd_size, f"too few tile CSVs for video '{v}'"
    G[v] = {u: fov_samples(parse_tile_csv(os.path.join(tile_dir, f"{v}_{u}_tile.csv")), CFG[v])[0] for u in U[v]}
    held = "U_test" in PRED_CK[v] and v not in PRED_OVERRIDE
    U_test[v] = list(PRED_CK[v]["U_test"]) if held else U[v][::5]

n_par = lambda m: sum(p.numel() for p in m.parameters())
print(f"grid {cfg.H} x {cfg.W}, order n = {cfg.n}, {cfg.n_samples} samples per viewer")
for v in pcfg.videos:
    print(f"  {v:10s} |U| = {len(U[v])}, predictor {os.path.basename(PRED_FILE[v])} ({n_par(PRED[v]):,} weights), "
          f"test crowd {U_test[v]}")
print(f"encoder from {os.path.basename(ENC_FILE)} ({n_par(enc):,} weights): L = {L}, hidden {ENC_HIDDEN}, "
      f"tau = {tuple(round(float(t), 4) for t in ENC_CK.get('tau', ()))}, "
      f"normalised = {NORMALISED}" + (f" (s = {NORM_SCALE:.4f})" if NORMALISED else "")
      + f", videos = {ENC_CK.get('videos', 'not listed')}")

## 4. Crowds and their predicted frames, per video

For every video, 200 random **10-viewer crowds** are drawn from its 50 dataset viewers, never exactly its test
crowd. For each crowd:

1. average its tile maps into the combined frames $F_1,\dots,F_{12}$;
2. run **that video's** frozen predictor on every window of five: $\hat F_k = f^{(v)}_\theta(F_{k-5},\dots,F_{k-1})$,
   $k = 6,\dots,12$;
3. keep the crowd's 10 true maps $G_{u,k}$: the **behaviours** that the video's headsets follow in an episode.

| set | per video | used for |
|---|---|---|
| **train** | 160 crowds | every training episode draws one crowd per watched video at random |
| **validation** | 40 crowds | 40 fixed validation episodes (fixed seeds) |
| **test** | the test crowd `U_test[v]` | 50 fixed test episodes in section 11 |

The cell also checks the common encoder on every video's test frames: the number of **active** latent slots (mean
$\sigma_z < 0.5$). A number much lower for some videos than for others suggests the encoder has not seen those
videos.

In [ ]:
rng_c = np.random.default_rng(pcfg.seed)


def crowd_frames(v, users):
    # F_hat_k for k = 6..12 from video v's frozen predictor, and the true maps G_{u,k} of the crowd
    X, _, ks = combined_pairs(combined(G[v], users))
    F_hat = predict(v, torch.from_numpy(X).to(DEVICE))                     # (7, H, W)
    G_true = np.stack([[G[v][u][k - 1] for k in ks] for u in users])       # (|crowd|, 7, H, W)
    return F_hat.astype(np.float32), G_true.astype(np.float32), ks


TRAIN, VAL, TEST = {}, {}, {}
for v in pcfg.videos:
    crowds = []
    while len(crowds) < pcfg.n_crowds:
        sub = sorted(rng_c.choice(U[v], size=pcfg.crowd_size, replace=False).tolist())
        if set(sub) != set(U_test[v]):
            crowds.append(sub)
    data = [crowd_frames(v, sub)[:2] for sub in crowds]
    val_ids = set(rng_c.choice(pcfg.n_crowds, size=int(round(pcfg.val_frac * pcfg.n_crowds)), replace=False).tolist())
    TRAIN[v] = [data[i] for i in range(pcfg.n_crowds) if i not in val_ids]
    VAL[v]   = [data[i] for i in sorted(val_ids)]
    TEST[v]  = [crowd_frames(v, U_test[v])[:2]]                           # a list of one crowd, like TRAIN and VAL
KS = list(range(cfg.n + 1, cfg.n_samples + 1))
N_SLOTS = len(KS)                                                          # 7 intervals per episode

print(f"per video: {pcfg.n_crowds} random {pcfg.crowd_size}-viewer crowds (never the test crowd): "
      f"train {len(TRAIN[pcfg.videos[0]])}, val {len(VAL[pcfg.videos[0]])}; intervals per episode: k = {KS}")
for v in pcfg.videos:
    with torch.no_grad():
        mu_t, lv_t = enc(torch.from_numpy(encoder_input(TEST[v][0][0])).to(DEVICE))
    sig_t = torch.exp(0.5 * lv_t).mean(0).cpu().numpy()
    print(f"  {v:10s} F_hat range over its train crowds [{min(f.min() for f, _ in TRAIN[v]):.3f}, "
          f"{max(f.max() for f, _ in TRAIN[v]):.3f}];  encoder on its test crowd: {(sig_t < 0.5).sum()} of {L} "
          f"latent slots active (mean sigma < 0.5)")

## 5. Quantile masks $M_l$, group sizes $T_l$ and mean attention $w_l$

### The formulas (methodology, page 1)

The range $[0,1]$ of a tile's predicted attention is cut into $L = 10$ equal intervals,

$$I_1 = \left[0,\ \tfrac{1}{L}\right],\qquad I_l = \left(\tfrac{l-1}{L},\ \tfrac{l}{L}\right],\quad l = 2,\dots,L,$$

and every tile is put in the group of its interval:

$$M_l(i,j) = \begin{cases}1 & \hat F_k(i,j) \in I_l\\ 0 & \text{otherwise}\end{cases},\qquad
T_l = \sum_{i=1}^{W}\sum_{j=1}^{H} M_l(i,j),\qquad
w_l = \frac{1}{T_l}\sum_{i=1}^{W}\sum_{j=1}^{H} \hat F_k(i,j)\,M_l(i,j).$$

Each tile is in exactly one group, so $\sum_l M_l(i,j) = 1$ and $\sum_l T_l = 200$. A tile in group $l$ is
**targeted at SVC layers $1,\dots,l$**: the base layer for everybody, more layers where more of the crowd looks.

**Empty groups.** If no tile falls in $I_l$ then $T_l = 0$, and $w_l$ is set to 0. Bandwidth given
to an empty group reaches no tile: it is wasted, so the agent learns to give empty groups nothing. (Group 10, "90-100% of the crowd", is empty in most frames.)

### Tiny example (4 tiles, $L = 4$)

$\hat F = [0.05,\ 0.30,\ 0.32,\ 0.90]$ → intervals $[0, .25], (.25, .5], (.5, .75], (.75, 1]$ → groups
$[1, 2, 2, 4]$, so $T = [1, 2, 0, 1]$ and $w = [0.05,\ 0.31,\ 0,\ 0.90]$. The tile at 0.90 is targeted at all 4
layers, the tile at 0.05 at the base layer only, and group 3 is empty.

**The cell below** prints $T_l$ and $w_l$ for the first test interval and shows its groups next to $\hat F_k$.

In [ ]:
def quantile_masks(F_hat, L=L):
    # group g(i,j) in 1..L with F_hat(i,j) in I_g;  M_l, T_l, w_l of the methodology
    grp = np.clip(np.ceil(F_hat * L).astype(int), 1, L)                     # (H, W)
    M = (grp[None] == np.arange(1, L + 1)[:, None, None]).astype(np.float32)  # (L, H, W)
    T = M.sum((1, 2))
    w = np.where(T > 0, (M * F_hat[None]).sum((1, 2)) / np.maximum(T, 1), 0.0)
    return M, grp, T, w


V0 = pcfg.videos[0]
F_ex = TEST[V0][0][0][0]                                                   # test crowd, first interval
M_ex, grp_ex, T_ex, w_ex = quantile_masks(F_ex)
assert np.all(M_ex.sum(0) == 1) and T_ex.sum() == cfg.H * cfg.W          # every tile in exactly one group
print(f"video '{V0}', first test interval:")
print(pd.DataFrame({"interval I_l": [f"[0, {1/L:.1f}]"] + [f"({(l-1)/L:.1f}, {l/L:.1f}]" for l in range(2, L + 1)],
                    "T_l": T_ex.astype(int), "w_l": w_ex.round(3)},
                   index=pd.Index(range(1, L + 1), name="l")).T.to_string())

for v in pcfg.videos:
    T_all = np.stack([quantile_masks(f)[2] for F_hat, _ in TRAIN[v] for f in F_hat])
    print(f"\n{v}: mean T_l over the train frames {T_all.mean(0).round(1)}")
    print(f"{'':{len(v)}s}  share of frames with group l empty {(T_all == 0).mean(0).round(2)}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 2.8))
im = ax1.imshow(F_ex, cmap="inferno", vmin=0, vmax=1, aspect="auto"); fig.colorbar(im, ax=ax1)
ax1.set_title(rf"predicted $\hat F_{{{KS[0]}}}$ ('{V0}', test crowd)")
im = ax2.imshow(grp_ex, cmap=plt.get_cmap("tab10", L), vmin=0.5, vmax=L + 0.5, aspect="auto")
fig.colorbar(im, ax=ax2, ticks=range(1, L + 1))
ax2.set_title(r"group $l$ of each tile ($M_l(i,j) = 1$)")
for ax in (ax1, ax2):
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "ddpg_masks.png"), dpi=130)
plt.show()

## 6. The radio channel and the path-loss statistics $\mu_k$, $\sigma_k$

The channel follows `main.pdf` (the model of Liang et al. [1], at $f_c = 3.5$ GHz).

### Channel gain
The downlink power gain from the edge server to headset $u$ in interval $k$ is a large-scale times a small-scale
part:

$$g_{u,k} = \alpha_{u,k}\,\tilde h_{u,k},\qquad \tilde h_{u,k} \sim \mathrm{Exp}(1)\ \ \text{(Rayleigh fading, unit mean)} .$$

### Large-scale path loss $h_{u,k}$ (in dB)

$$h_{u,k} = -10\log_{10}\alpha_{u,k}
= \underbrace{128.1 - 37.6\log_{10}\!\Big(\frac{1\,\text{km}}{d_0}\Big)}_{\text{constant}\ =\ 15.3\ \text{dB for}\ d_0 = 1\,\text{m}}
+ \underbrace{37.6\log_{10}\!\Big(\frac{d_{u,k}}{d_0}\Big)}_{\text{distance}}
+ \underbrace{20\log_{10}\!\Big(\frac{f_c}{2\,\text{GHz}}\Big)}_{\text{carrier}\ \approx\ 4.86\ \text{dB}}
+ \underbrace{X_{u,k}}_{\text{shadowing}} .$$

This is $128.1 + 37.6\log_{10} d_\text{km}$ of [1] rewritten for metres ($\log_{10}(d/1\,\text{km}) = \log_{10}(d/1\,\text{m}) - 3$),
plus a free-space frequency term because [1] is specified at 2 GHz. The constant is a fitted value, not a real
loss at 1 m: the model is used for $d_{u,k} \ge 35$ m.

### Placing and moving the headsets
At the start of an episode each of the 50 headsets is put at a random spot in the ring $35 \le d \le 200$ m,
uniformly over its area, with a random walking direction and a shadowing value $X_{u,6} \sim \mathcal N(0, 8^2)$ dB.
Every interval each headset walks $1.5\ \text{m/s} \times 5\ \text{s} = 7.5$ m (it turns back at the edges of the
ring), and the shadowing follows it with the 50-m decorrelation distance (Gudmundson model):

$$X_{u,k} = \rho\,X_{u,k-1} + \sqrt{1-\rho^2}\;\xi\,\mathcal N(0,1),\qquad \rho = e^{-7.5/50} = 0.86 .$$

The large-scale loss is held constant within one interval (the PDF updates it every 100 ms; over one interval it
changes by little).

### SNR and the rate over a segment
Video $v$ gets power $P_v$ and band $B_v$ (section 7), spread evenly, so every hertz of its band sees the same mean
SNR (Eq. (3) of the PDF):

$$\bar\gamma_{u,k} = \frac{P_v\,G_t\,G_r\,10^{-h_{u,k}/10}}{N_0\,F\,B_v}.$$

When every video's power is in proportion to its band ($P_v/B_v = P/B_\text{tot}$), this equals the SNR of the whole
power over the whole band, $N_0 F B_\text{tot} = -92$ dBm at 20 MHz. The figure and the percentiles below use that
case.

The Rayleigh fading $\tilde h_{u,k}$ changes every 1 ms, so a 5-s segment sees thousands of fades and its rate is
the fading-averaged (ergodic) one:

$$\mathrm{SE}_{u,k} = \mathbb E_{\tilde h}\big[\log_2\big(1 + \bar\gamma_{u,k}\,\tilde h\big)\big]
= \int_0^\infty e^{-x}\log_2(1 + \bar\gamma_{u,k}\,x)\,dx\quad\text{bit/s/Hz},$$

which has the closed form $e^{1/\bar\gamma}E_1(1/\bar\gamma)/\ln 2$ ($E_1$ is the exponential integral).

### $\mu_k$ and $\sigma_k$ (methodology, page 2)
Exponentially weighted moving averages, **per video**, of the large-scale path loss of the headsets watching it $h_{u,k}$ (in dB; the fast fading
only affects the instantaneous SNR and is left out), started at the first interval with the plain sample mean and
mean absolute deviation:

$$\mu_k = (1-\alpha)\,\mu_{k-1} + \frac{\alpha}{|\mathcal U|}\sum_{u\in\mathcal U}h_{u,k},\qquad
\sigma_k = (1-\beta)\,\sigma_{k-1} + \frac{\beta}{|\mathcal U|}\sum_{u\in\mathcal U}\big|h_{u,k}-\mu_k\big| .$$

**Tiny example.** $\mu_{k-1} = 95$ dB, $\sigma_{k-1} = 8$ dB, $\alpha = \beta = 0.3$, this interval's path losses
$[90, 100, 104]$ (mean 98): $\mu_k = 0.7\cdot 95 + 0.3\cdot 98 = 95.9$; deviations from 95.9 are
$[5.9, 4.1, 8.1]$ (mean 6.03), so $\sigma_k = 0.7 \cdot 8 + 0.3 \cdot 6.03 = 7.41$.

**The figure** shows the loss against distance for many random headsets (as Fig. 2 of the PDF), their mean SNR,
and the Rayleigh fading histogram.

In [ ]:
def place_users(n, rng, p=pcfg):
    # positions uniform over the area of the ring d_min..radius, a walking heading, and the shadowing X_u (dB)
    d = np.sqrt(rng.uniform(p.d_min_m ** 2, p.radius_m ** 2, n))
    phi = rng.uniform(0.0, 2 * np.pi, n)
    pos = d[:, None] * np.stack([np.cos(phi), np.sin(phi)], 1)
    return pos, rng.uniform(0.0, 2 * np.pi, n), rng.normal(0.0, p.shadow_db, n)


def move_users(pos, head, X, rng, p=pcfg):
    # walk walk_mps * T_seg along the heading, turning back at the edges of the ring; shadowing correlated over
    # the distance walked (Gudmundson: rho = exp(-step / d_corr))
    step = p.walk_mps * p.T_seg
    dxy = step * np.stack([np.cos(head), np.sin(head)], 1)
    new = pos + dxy
    r = np.linalg.norm(new, axis=1)
    out = (r < p.d_min_m) | (r > p.radius_m)
    new[out], head = pos[out] - dxy[out], np.where(out, head + np.pi, head)
    r = np.linalg.norm(new, axis=1)
    new *= (np.clip(r, p.d_min_m, p.radius_m) / r)[:, None]
    rho = np.exp(-step / p.shadow_dcorr_m)
    return new, head, rho * X + np.sqrt(1 - rho ** 2) * rng.normal(0.0, p.shadow_db, len(X))


def path_loss_db(d_m, p=pcfg):
    # distance-only large-scale path loss of [1] in dB, d in m (d >= 35 m), at carrier f_c
    return (128.1 - 37.6 * np.log10(1000.0 / p.d0_m)                    # constant: 15.3 dB for d0 = 1 m
            + 37.6 * np.log10(d_m / p.d0_m)                              # distance-dependent path loss
            + 20 * np.log10(p.f_c_hz / 2e9))                             # carrier-frequency correction


def ergodic_se(snr, n=40):
    # E[log2(1 + snr h~)], h~ ~ Exp(1): bit/s/Hz averaged over the Rayleigh fading of one segment,
    # = e^z E1(z) / ln 2 with z = 1/snr (E1: exponential integral; series for z <= 2, continued fraction above)
    z = 1.0 / np.asarray(snr, float)
    k = np.arange(1, n + 1)
    zs = np.minimum(z, 2.0)
    series = -np.euler_gamma - np.log(zs) - ((-zs[..., None]) ** k / (k * np.cumprod(k.astype(float)))).sum(-1)
    f = z + 2 * n + 1
    for j in range(n, 0, -1):
        f = z + 2 * j - 1 - j * j / f
    return np.where(z <= 2.0, np.exp(zs) * series, 1.0 / f) / np.log(2)


def channel_step(d, X, p=pcfg):
    # h_{u,k} = -10 log10 alpha_{u,k} (dB), mean SNR gamma_{u,k} over the band, and the segment rate per Hz
    h_db = path_loss_db(d, p) + X
    snr = p.P_t_W * 10 ** ((p.gain_db - h_db) / 10) / p.noise_W
    return h_db, snr, ergodic_se(snr)


def ewma(mu_prev, sig_prev, pl, p=pcfg):
    # mu_k, sigma_k of the methodology; the first interval starts from the sample statistics
    if mu_prev is None:
        mu = pl.mean()
        return mu, np.abs(pl - mu).mean()
    mu = (1 - p.ewma_a) * mu_prev + p.ewma_a * pl.mean()
    return mu, (1 - p.ewma_b) * sig_prev + p.ewma_b * np.abs(pl - mu).mean()


rng_ch = np.random.default_rng(pcfg.seed)
pos_s, _, X_s = place_users(20000, rng_ch)
d_s = np.linalg.norm(pos_s, axis=1)
h_s, snr_s, se_s = channel_step(d_s, X_s)
fad_db = 10 * np.log10(rng_ch.exponential(1.0, 200000))                 # 10 log10 h~, Rayleigh power gain
g_db = -h_s + fad_db[:len(h_s)]                                         # 10 log10 g_{u,k}, one fade per headset
print(f"carrier term 20 log10(f_c / 2 GHz) = {20 * np.log10(pcfg.f_c_hz / 2e9):.2f} dB,  "
      f"noise N0 F B_tot = {10 * np.log10(pcfg.noise_W) + 30:.1f} dBm")
print(f"Rayleigh 10log10 h~: mean {fad_db.mean():.2f} dB, std {fad_db.std():.2f} dB")
print(f"path loss h_uk     5/50/95%: {np.percentile(h_s, [5, 50, 95]).round(1)} dB")
print(f"mean SNR           5/50/95%: {np.percentile(10 * np.log10(snr_s), [5, 50, 95]).round(1)} dB")
print(f"E log2(1+SNR h~)   5/50/95%: {np.percentile(se_s, [5, 50, 95]).round(2)} bit/s/Hz  -> "
      f"whole band {(pcfg.B_tot * np.percentile(se_s, [5, 50, 95]) / 1e6).round(0)} Mbit/s")

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
dd = np.linspace(pcfg.d_min_m, pcfg.radius_m, 200)
ax = axes[0]
ax.fill_between(dd, path_loss_db(dd) - pcfg.shadow_db, path_loss_db(dd) + pcfg.shadow_db, color="#1f77b4",
                alpha=.12, label=rf"$\pm\xi$ = {pcfg.shadow_db:.0f} dB shadowing band")
ax.scatter(d_s[:1500], -g_db[:1500], s=3, alpha=.4, color="#ff7f0e", label=r"with fading $-10\log_{10} g_{u,k}$")
ax.scatter(d_s[:1500], h_s[:1500], s=3, alpha=.4, color="#1f77b4", label=r"path loss + shadowing $h_{u,k}$")
ax.plot(dd, path_loss_db(dd), color="k", label=r"distance only (no $X_{u,k}$)")
ax.set_xlabel(r"distance $d_{u,k}$ (m)"); ax.set_ylabel("loss (dB)"); ax.legend(fontsize=7)
ax.set_title(f"Path loss vs distance at f_c = {pcfg.f_c_hz / 1e9:.1f} GHz")
axes[1].scatter(d_s[:3000], 10 * np.log10(snr_s[:3000]), s=3, alpha=.3, color="#2ca02c")
axes[1].set_xlabel(r"distance $d_{u,k}$ (m)"); axes[1].set_ylabel("mean SNR (dB)")
axes[1].set_title(r"$\bar\gamma_{u,k} = P G_t G_r 10^{-h_{u,k}/10} / (N_0 F B_\mathrm{tot})$")
axes[2].hist(fad_db, bins=100, color="#9467bd")
axes[2].set_xlabel(r"$10\log_{10}\tilde h_{u,k}$ (dB)"); axes[2].set_title(r"Rayleigh fading, $\tilde h \sim \mathrm{Exp}(1)$")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "ddpg_channel.png"), dpi=130)
plt.show()

## 7. The environment: from an action to a reward

`StreamingEnv(split, seed)` plays one episode. `reset()` returns the first state; `step(a)` serves one video and
returns the reward, the next state, and whether the episode is over.

### An episode
1. **Headsets.** 50 headsets are placed in the ring (section 6). Each picks a video $v$ with probability $p_v$
   (Zipf). $\mathcal A_v$ is the set of headsets on video $v$, $N_v = |\mathcal A_v|$, and the **watched** videos are
   those with $N_v > 0$, served in rank order.
2. **Behaviours.** For each watched video one 10-viewer crowd is drawn from the split. Its predicted frames
   $\hat F^{(v)}_k$ drive the masks and the state. Each headset in $\mathcal A_v$ follows one of the crowd's 10
   behaviours, drawn with replacement: that is its true FoV $G_{u,k}$.
3. **Intervals.** For $k = 6,\dots,12$: the headsets walk, the path losses change, and the whole band and power are
   free again ($B_\text{rem} = B_\text{tot}$, $P_\text{rem} = P_\text{tot}$). Then there is **one step per watched
   video**, in rank order.

### The state of a step (28 numbers)

$$S = \Big[\ \mathrm{sigmoid}(z^{(v)}_k),\ w^{(v)}_k,\ \tfrac{\mu^{(v)}_k - 100}{10},\ \tfrac{\sigma^{(v)}_k}{10},\ \bar q^{(v)},\ q^{(v),\min},\ \tfrac{B_\text{rem}}{B_\text{tot}},\ \tfrac{P_\text{rem}}{P_\text{tot}},\ \tfrac{N_v}{50},\ \tfrac{\#\text{watched videos after } v}{V}\ \Big]$$

$\mu^{(v)}$ and $\sigma^{(v)}$ are the moving averages of the path loss of the headsets in $\mathcal A_v$, and $\bar
q$, $q^{\min}$ are their mean and smallest playout buffer. The last two numbers were added (assumed): without them
the actor cannot tell how much to keep for the videos still to come.

### From the action to bandwidth and power

$$a = [\,b_0,\ b_1,\dots,b_L,\ \rho\,],\qquad b = \mathrm{softmax}(\cdot)\ \ (\textstyle\sum_{l=0}^{L} b_l = 1),\qquad \rho = \mathrm{sigmoid}(\cdot) \in (0,1).$$

The video takes $B_v = (1 - b_0)\,B_\text{rem}$ of the free band, split over its groups, and $P_v = \rho\,P_\text{rem}$
of the free power:

$$B(i,j) = \sum_{l=1}^{L} M_l(i,j)\,\frac{b_l\,B_\text{rem}}{T_l},\qquad B_\text{rem} \leftarrow B_\text{rem} - B_v,\qquad P_\text{rem} \leftarrow P_\text{rem} - P_v .$$

**The last watched video of the interval takes everything left**: the environment sets $b_0 = 0$ (rescaling
$b_1..b_L$) and $\rho = 1$. This overridden action is the one stored in the replay buffer. So every interval uses
the whole band and the whole power, and the cell checks this. Bandwidth given to an empty group reaches no tile:
it is wasted.

### SNR and the rate of a headset
Video $v$'s power is spread evenly over its band (flat PSD), so its headsets see

$$\bar\gamma_{u} = \frac{P_v\,G_t\,G_r\,10^{-h_{u,k}/10}}{N_0\,F\,B_v},\qquad R_u(i,j) = B(i,j)\,\mathbb E\big[\log_2(1 + \bar\gamma_u\tilde h)\big].$$

Giving a video more power than its share of the band raises its SNR and lowers the SNR left for the others. If
every video takes power in proportion to its bandwidth, the SNR is the same as with one video on the whole band.

### Layers, playout queue and freezes
These are unchanged from the single-video version. Viewer $u$ gets the most layers that fit in the segment and do
not exceed the tile's target:

$$n_u(i,j) = \max\{m \le l(i,j) : T_\text{seg}R_m / R_u(i,j) \le T_\text{seg}\}.$$

The segment is played as five 1-s chunks. The base layer of chunk $m$ arrives at $a_{u,m} = ((m-1)R_n + R_1)/R_u$,
and is due at $p_{u,m}$, with $p_{u,1}$ = the viewer's buffer and $p_{u,m+1} = p_{u,m} + 1\,\text{s} +
\text{stall}_{u,m}$. A late chunk freezes playback for the $\hat F$-weighted mean lateness of the tiles the viewer
receives, and the buffer carries the freezes over: $q_{u,k+1} = q_{u,k} + T_{u,k}$. A tile that receives
nothing ($n = 0$) does not freeze playback by itself. But **a viewer who receives nothing on any tile cannot play
at all**, so their freeze is the whole segment, $T_{u,k} = 5$ s. Without this rule, starving a video completely
would cost nothing ($Q = 0$, $T = 0$), and the agent could learn to drop unpopular videos to avoid their freeze
penalties.

### The reward: Seer's QoE
For each headset $u$ in interval $k$:

$$R_{u,k} = \lambda\,Q_{u,k} - \mu\,T_{u,k} - \xi\,\big|Q_{u,k} - Q_{u,k-1}\big|$$

(the last term is 0 in the headset's first interval, as Seer's sum starts at $k = 2$). The terms:
* **$T_{u,k}$, rebuffering** = the viewer's total freeze in the segment, in seconds: $\sum_m \text{stall}_{u,m}$.
* **$q_{u,k}$, bitrate.** Seer gives all tiles of a segment one bitrate; here each tile has its own number of
  layers. The tiles are weighted by predicted attention and scaled to the whole frame:
  $$q_{u,k} = H W \sum_{i,j}\hat w(i,j)\,R_{n_u(i,j)},\qquad \hat w = \hat F_k / \textstyle\sum\hat F_k .$$
  It is the bitrate of a uniform video that looks as good where people look. When every tile has $m$ layers,
  $q = 200\,R_m$, exactly Seer's $q$. A tile that receives nothing adds 0.
* **$Q$**, `q_mode` (**open**):
  * option A, `"normalised"`: $Q = q / Q_\text{max}$, with $Q_\text{max} = 200\,R_{10}$ (160 Mbit/s at 8K, 40 at
    4K), $\lambda = \xi = 1$, $\mu = 1$.
  * option B, `"mbps"`: $Q = q$ in Mbit/s, $\lambda = \xi = 1$, $\mu = Q_\text{max}$ in Mbit/s (160 at 8K).

  Option B is option A times $Q_\text{max}$, so both have the same optimal policy; A keeps the rewards near 1 for
  the critic. In both, $\mu = Q_\text{max}$ in the units of $Q$, as in Seer ($\mu = 4.3$ = its top $Q$).

A video's step reward is the mean over its headsets, weighted by its share of the 50:

$$\boxed{\,r^{(v)}_k = \frac{N_v}{50}\cdot\frac{1}{N_v}\sum_{u\in\mathcal A_v} R_{u,k}\,}\qquad\Rightarrow\qquad \sum_{v} r^{(v)}_k = \frac{1}{50}\sum_{u} R_{u,k}.$$

**Only reported** (not in the reward): $Q$ on the viewer's **true** map $G_{u,k}$ (`Q_true`), $q$ in Mbit/s, and the
earlier utilities $U_b$ (log bitrate) and $U_d$ (mean $e^{-\text{lateness}}$ per chunk).

**Tiny example** (one tile, one viewer, 4K ladder, $D_0 = 0.5$ s): $R_u = 21$ kbit/s (2.8 kHz × SE 7.5). With $R_1,
R_2, R_3 = 13.8, 19.0, 25.9$ kbit/s, $t_1, t_2, t_3 = 3.29, 4.52, 6.17$ s, so $n = 2$. The base of chunk 1 arrives at
$13.8/21 = 0.657$ s but is due at 0.5 s, so playback freezes for $T = 0.157$ s. Chunks 2-5 arrive on time, since
each chunk takes $19.0/21 = 0.905$ s, less than 1 s. Under option A, $Q = 19.0/200 = 0.095$ and, in the first
interval, $R = 0.095 - 0.157 = -0.062$: the freeze outweighs the quality.

**The cell below** prints the 4K and 8K ladders, the demand against capacity in every scenario, the delay checks,
and one interval of the test set.

In [ ]:
N_CHUNK = int(round(pcfg.T_seg / pcfg.chunk_s))  # 5 chunks per segment
HW = cfg.H * cfg.W                               # 200 tiles


def rates(p=pcfg):
    # the SVC ladder of one scenario (its video_px): per tile, and Q_max = all layers on all tiles (bit/s)
    tile_px = (p.video_px[0] // cfg.W) * (p.video_px[1] // cfg.H)
    R_L, QP = cumulative_rates(L, tile_px, p)
    return SimpleNamespace(tile_px=tile_px, R_L=R_L, QP=QP, R_LAYER=np.diff(R_L, prepend=0.0),
                           S_CUM=p.T_seg * R_L,                          # S_m: bits of layers 1..m in one segment
                           R_CUM=np.concatenate([[0.0], R_L]),          # bitrate with n = 0..L layers
                           Q_max=HW * R_L[-1])


SCEN_RT = {name: rates(c) for name, c in SCEN_CFG.items()}
RT = SCEN_RT[BASE_SCEN]


def seer_weights(p, rt):
    # Seer, Eq. (6): R = lam Q - mu T - xi |Q - Q_prev|.  Returns (the bit/s that make one unit of Q, lam, mu, xi)
    unit = {"normalised": rt.Q_max, "mbps": 1e6}[p.q_mode]
    mu = rt.Q_max / unit if p.mu_rebuf is None else p.mu_rebuf     # Seer: mu = the top Q (4.3 there)
    return unit, p.lam_q, mu, p.xi_smooth


# ---- utilities that are reported (no longer in the reward) ----
def u_bitrate(n, rt):  return np.log1p(rt.R_CUM[n] / rt.R_CUM[1]) / np.log1p(rt.R_CUM[-1] / rt.R_CUM[1])
def u_delay(late):     return np.exp(-late)                  # late in s; 1 when on time (late = 0)


def allocate(M, T, b, B_avail):
    # B(i,j) = sum_l M_l b_l B_avail / T_l   (b: shares of the bandwidth still free; their sum is the video's part)
    return np.tensordot(np.where(T > 0, b * B_avail / np.maximum(T, 1), 0.0), M, 1)


def snr_at(h_db, P_W, B_hz, p=pcfg):
    # mean SNR of the headsets of one video: its power P_W spread evenly over its band B_hz (flat PSD)
    return P_W * 10 ** ((p.gain_db - h_db) / 10) / p.noise_at(B_hz)


def delay_utility(n, R, wF, buf, rt, p=pcfg):
    """Per-chunk lateness with a playout queue per viewer.

    n: (|U|, H, W) layers received, R: (|U|, H, W) bit/s, wF: (H, W) tile weights summing to 1, buf: (|U|,) s.
    Returns the delay utility per viewer, the total freeze per viewer (Seer's T), the buffer for the next segment,
    the delay utility of each chunk (mean over viewers) and of each tile (mean over viewers and chunks)."""
    got = n > 0
    Rs = np.where(got, R, 1.0)
    x = np.where(got, rt.R_CUM[n] * p.chunk_s / Rs, 0.0)       # s to send one chunk of the n layers
    base = np.where(got, rt.R_CUM[1] * p.chunk_s / Rs, 0.0)    # s to send the base layer of one chunk
    w_got = wF[None] * got
    w_sum = np.maximum(w_got.sum((1, 2)), 1e-12)
    due = np.asarray(buf, float).copy()                        # p_{u,1}: chunk 1 is due when the buffer runs out
    u_sum, stall_tot, per_chunk = np.zeros(n.shape), np.zeros(len(due)), []
    for m in range(1, N_CHUNK + 1):
        arrive = (m - 1) * x + base                            # a_{u,m}(i,j)
        late = np.where(got, np.maximum(0.0, arrive - due[:, None, None]), 0.0)
        u = np.where(got, u_delay(late), 0.0)                  # a tile that receives nothing scores 0
        stall = (late * w_got).sum((1, 2)) / w_sum             # F_hat-weighted mean lateness -> freeze
        u_sum += u
        stall_tot += stall
        per_chunk.append(float((u * wF).sum((1, 2)).mean()))
        due = due + p.chunk_s + stall                          # p_{u,m+1} = p_{u,m} + 1 s + stall_{u,m}
    blank = ~got.any((1, 2))                                   # a viewer who receives nothing at all cannot play:
    stall_tot = np.where(blank, p.T_seg, stall_tot)            # the whole segment is a freeze
    due = np.where(blank, due + p.T_seg, due)
    Ud_u = (u_sum / N_CHUNK * wF).sum((1, 2))
    buf_next = due - p.T_seg if p.carry_queue else np.full(len(due), p.D0_s)   # = buf + total freeze
    return Ud_u, stall_tot, buf_next, per_chunk, (u_sum / N_CHUNK).mean(0)


def serve(F_hat, grp, B, se, buf, G_true, rt, p=pcfg):
    # one video: layers n_u(i,j) each of its headsets receives; per headset Seer's bitrate q (whole-frame
    # equivalent, bit/s) on the predicted and on the true FoV, the freeze T, and the reported utilities
    R = B[None] * se[:, None, None]                                         # R_u(i,j), (|U|, H, W)
    with np.errstate(divide="ignore"):
        t = rt.S_CUM / R[..., None]                                          # t_{u,m}(i,j), (|U|, H, W, L)
    fits = (t <= p.T_seg) & (np.arange(1, L + 1) <= grp[None, ..., None])
    n = fits.sum(-1)                                     # t grows with m, so the fitting layers are 1..n
    wF = F_hat / F_hat.sum()
    wG = G_true / np.maximum(G_true.sum((1, 2), keepdims=True), 1e-12)
    Ud_u, stall, buf_next, Ud_chunk, Ud_tile = delay_utility(n, R, wF, buf, rt, p)
    return {"n": n, "q_u": HW * (rt.R_CUM[n] * wF).sum((1, 2)), "q_true_u": HW * (rt.R_CUM[n] * wG).sum((1, 2)),
            "Ub_u": (u_bitrate(n, rt) * wF).sum((1, 2)), "Ub_true_u": (u_bitrate(n, rt) * wG).sum((1, 2)),
            "Ud_u": Ud_u, "stall_u": stall, "buf_next": buf_next, "Ud_chunk": Ud_chunk, "Ud_tile": Ud_tile}


class StreamingEnv:
    """One episode. n_users headsets pick videos by Zipf; 7 intervals k = 6..12; in every interval one step per
    watched video, in rank order, sharing the band B_tot and the power P_t. split: TRAIN, VAL or TEST."""

    def __init__(self, split, seed, p=pcfg, rt=None, record=False):
        self.split, self.p, self.record = split, p, record
        self.rt = rt if rt is not None else rates(p)
        self.rng = np.random.default_rng(seed)
        self.unit, self.lam, self.mu_r, self.xi = seer_weights(p, self.rt)

    def reset(self):
        p, rng = self.p, self.rng
        self.pos, self.head, self.X = place_users(p.n_users, rng, p)
        self.choice = rng.choice(V_N, size=p.n_users, p=P_ZIPF)                 # the video of every headset
        self.active = [v for v in range(V_N) if np.any(self.choice == v)]        # watched videos, rank order
        self.users = {v: np.flatnonzero(self.choice == v) for v in self.active}  # A_v
        self.F_hat = {}
        self.G_true = np.zeros((p.n_users, N_SLOTS, cfg.H, cfg.W), np.float32)
        for v in self.active:
            crowds = self.split[p.videos[v]]
            F, Gt = crowds[rng.integers(len(crowds))]                           # one 10-viewer crowd of video v
            self.F_hat[v] = F
            self.G_true[self.users[v]] = Gt[rng.integers(0, len(Gt), len(self.users[v]))]   # its 10 behaviours
        self.buf = np.full(p.n_users, p.D0_s)                                   # playout buffer q_{u,6} = D_0
        self.Q_prev = np.full(p.n_users, np.nan)
        self.mu, self.sig = {v: None for v in self.active}, {v: None for v in self.active}
        self.t, self.j, self.history = 0, 0, []
        self._new_interval()
        self._observe()
        return self.state

    def _new_interval(self):
        # the headsets walk (not before the first interval), new path losses; the whole band and power are free
        p = self.p
        if self.t > 0:
            self.pos, self.head, self.X = move_users(self.pos, self.head, self.X, self.rng, p)
        self.h_db = path_loss_db(np.linalg.norm(self.pos, axis=1), p) + self.X  # h_{u,k}
        self.B_rem, self.P_rem, self.B_used, self.P_used = p.B_tot, p.P_t_W, 0.0, 0.0
        self.masks, self.z = {}, {}
        for v in self.active:
            self.mu[v], self.sig[v] = ewma(self.mu[v], self.sig[v], self.h_db[self.users[v]], p)
            self.masks[v] = quantile_masks(self.F_hat[v][self.t])
            self.z[v] = z_for_policy(self.F_hat[v][self.t])[0]                 # sigmoid(z_k), fresh draw

    # ---- the video served by the current step, and what is left after it ----
    @property
    def v(self):          return self.active[self.j]
    @property
    def N_v(self):        return len(self.users[self.v])
    @property
    def n_left(self):     return len(self.active) - self.j                     # this video and the ones after it
    @property
    def N_left_sum(self): return sum(len(self.users[v]) for v in self.active[self.j:])

    def _observe(self):
        p, v = self.p, self.v
        u = self.users[v]
        self.M, self.grp, self.T, self.w = self.masks[v]
        self.state = np.concatenate([self.z[v], self.w,
                                     [(self.mu[v] - p.pl_ref_db) / p.pl_scale_db, self.sig[v] / p.pl_scale_db,
                                      self.buf[u].mean(), self.buf[u].min(),
                                      self.B_rem / p.B_tot, self.P_rem / p.P_t_W,
                                      len(u) / p.n_users, (self.n_left - 1) / V_N]]).astype(np.float32)

    def step(self, a):
        p, rt, v = self.p, self.rt, self.v
        u = self.users[v]
        a = np.asarray(a, np.float64).copy()
        if self.n_left == 1:                       # the last watched video takes all that is left
            a[1:L + 1] /= a[1:L + 1].sum(); a[0] = 0.0; a[L + 1] = 1.0
        self.applied_action = a.astype(np.float32)
        b, rho = a[1:L + 1], a[L + 1]
        B_v, P_v = float(b.sum() * self.B_rem), float(rho * self.P_rem)
        self.B = allocate(self.M, self.T, b, self.B_rem)
        se = ergodic_se(np.maximum(snr_at(self.h_db[u], P_v, max(B_v, 1.0), p), 1e-12))   # 1e-12: no power
        o = serve(self.F_hat[v][self.t], self.grp, self.B, se, self.buf[u], self.G_true[u, self.t], rt, p)
        Q, Q_true = o["q_u"] / self.unit, o["q_true_u"] / self.unit
        dQ = np.where(np.isnan(self.Q_prev[u]), 0.0, np.abs(Q - self.Q_prev[u]))   # 0 at a headset's first interval
        R_u = self.lam * Q - self.mu_r * o["stall_u"] - self.xi * dQ             # Seer's QoE of every headset
        r = len(u) / p.n_users * float(R_u.mean())
        terms = {"k": KS[self.t], "video": v, "N_v": len(u), "r": r, "QoE": float(R_u.mean()),
                 "Q": float(Q.mean()), "q_mbps": float(o["q_u"].mean() / 1e6), "T": float(o["stall_u"].mean()),
                 "dQ": float(dQ.mean()), "Q_true": float(Q_true.mean()), "Ub": float(o["Ub_u"].mean()),
                 "Ub_true": float(o["Ub_true_u"].mean()), "Ud": float(o["Ud_u"].mean()),
                 "buf": float(self.buf[u].mean()), "Bsh": B_v / p.B_tot, "Psh": P_v / p.P_t_W}
        if self.record:
            self.history.append({"t": self.t, "F": self.F_hat[v][self.t], "B": self.B, "n_mean": o["n"].mean(0),
                                 "Ud_tile": o["Ud_tile"], "Ud_chunk": o["Ud_chunk"], **terms})
        self.Q_prev[u], self.buf[u], self.last = Q, o["buf_next"], o
        self.B_rem -= B_v; self.P_rem -= P_v; self.B_used += B_v; self.P_used += P_v
        self.j += 1
        done = False
        if self.j == len(self.active):             # the interval is over
            assert np.isclose(self.B_used, p.B_tot) and np.isclose(self.P_used, p.P_t_W), "band or power not all used"
            self.j, self.t = 0, self.t + 1
            done = self.t == N_SLOTS
            if not done:
                self._new_interval()
        if not done:
            self._observe()
        return self.state, r, done, terms


# ---- the layer bitrates at 4K and 8K ----
for res in ("4K", "8K"):
    rt_ = SCEN_RT[f"{res}-20MHz"]
    side = int(np.sqrt(rt_.tile_px))
    print(f"{res}: tile {side} x {side} px;  base layer on all tiles {HW * rt_.R_L[0] / 1e6:.1f} Mbit/s, "
          f"all layers on all tiles (Q_max) {rt_.Q_max / 1e6:.0f} Mbit/s")
    print(pd.DataFrame({"QP": rt_.QP.astype(int), "R_k cumulative (kbit/s)": (rt_.R_L / 1e3).round(1),
                        "r_k layer alone (kbit/s)": (rt_.R_LAYER / 1e3).round(1)},
                       index=pd.Index(range(1, L + 1), name="layer k")).T.to_string(), "\n")

# ---- demand against capacity, every scenario ----
F_all = [f for v in pcfg.videos for F_hat, _ in TRAIN[v] for f in F_hat]
act_n = [(np.bincount(rng_ch.choice(V_N, pcfg.n_users, p=P_ZIPF), minlength=V_N) > 0).sum() for _ in range(5000)]
print(f"videos watched per episode: mean {np.mean(act_n):.2f} (expected {EXP_ACTIVE:.2f}), "
      f"distribution {dict(zip(*np.unique(act_n, return_counts=True)))}")
dem_rows = {}
for name, c in SCEN_CFG.items():
    rt_ = SCEN_RT[name]
    dem = np.mean([rt_.S_CUM[quantile_masks(f)[1] - 1].sum() / c.T_seg for f in F_all])   # bit/s for one video
    cap5, cap50 = c.B_tot / EXP_ACTIVE * np.percentile(se_s, [5, 50])                       # equal split, flat PSD
    dem_rows[name] = {"target layers, one video (Mbit/s)": dem / 1e6, "capacity per video, 5% headset": cap5 / 1e6,
                      "capacity per video, median headset": cap50 / 1e6, "demand / 5% capacity": dem / cap5}
print("\ndemand of one video's target layers against what an equal share of the band carries:")
print(pd.DataFrame(dem_rows).T.round(2).to_string())

# ---- the delay checks on the tiny example of the text (4K ladder) ----
rt4 = SCEN_RT["4K-20MHz"]
one = lambda x: np.full((1, 1, 1), x)
R_ex = 21e3                                                                  # B = 2.8 kHz x SE 7.5
t_ex = rt4.S_CUM[:3] / R_ex
assert t_ex[1] <= pcfg.T_seg < t_ex[2]                                       # n = 2 layers fit
Ud_ex, st_ex, q_ex, _, _ = delay_utility(one(2), one(R_ex), np.ones((1, 1)), np.array([0.5]), rt4)
assert np.isclose(st_ex[0], rt4.R_CUM[1] / R_ex - 0.5) and np.isclose(q_ex[0], 0.5 + st_ex[0])   # only chunk 1 late
Q_ex = rt4.R_CUM[2] / rt4.R_CUM[-1]                                          # one tile: Q = R_2 / R_L (option A)
print(f"\ntiny example: t = {t_ex.round(2)} s -> n = 2;  freeze {st_ex[0]:.3f} s, next buffer {q_ex[0]:.3f} s, "
      f"U_d = {Ud_ex[0]:.3f};  Seer (option A, first interval): R = {Q_ex:.3f} - {st_ex[0]:.3f} = {Q_ex - st_ex[0]:+.3f}")
Ud_ok, st_ok, _, _, _ = delay_utility(one(1), one(1e9), np.ones((1, 1)), np.array([0.5]), rt4)
assert np.isclose(Ud_ok[0], 1.0) and st_ok[0] == 0.0                        # everything on time
R_late = rt4.R_CUM[1] / 0.8                                                  # base of chunk 1 arrives at 0.8 s
Ud_l, st_l, q_l, _, _ = delay_utility(one(1), one(R_late), np.ones((1, 1)), np.array([0.5]), rt4)
assert np.isclose(st_l[0], 0.3) and np.isclose(Ud_l[0], (np.exp(-0.3) + 4) / 5) and np.isclose(q_l[0], 0.8)
_, _, q_reset, _, _ = delay_utility(one(1), one(R_late), np.ones((1, 1)), np.array([0.5]), rt4,
                                    replace(pcfg, carry_queue=False))
assert np.isclose(q_reset[0], pcfg.D0_s)
_, st_0, q_0, _, _ = delay_utility(one(0), one(0.0), np.ones((1, 1)), np.array([0.5]), rt4)
assert st_0[0] == pcfg.T_seg and np.isclose(q_0[0], 0.5 + pcfg.T_seg)       # nothing received -> 5-s freeze
print("delay checks passed (on time -> no freeze; 0.3 s late -> freeze 0.3 s; buffer carried over / reset; "
      "nothing received -> the whole segment frozen)")

# ---- one interval of the test set with 'attention-proportional, equal split across videos' ----
env = StreamingEnv(TEST, seed=0)
s = env.reset()
assert s.shape == (2 * L + 8,) and np.all(np.isfinite(s))
print(f"\nheadsets per video (rank order): {dict((pcfg.videos[v], len(env.users[v])) for v in env.active)}")
print(f"state of the first step ({s.size} numbers): {s.round(3)}")
for _ in range(len(env.active)):
    att = env.T * env.w
    f = 1.0 / env.n_left
    _, r, done, terms = env.step(np.concatenate([[1 - f], f * att / att.sum(), [f]]))
    print(f"  {pcfg.videos[terms['video']]:10s} N_v = {terms['N_v']:2d}  B share {terms['Bsh']:.3f}  "
          f"P share {terms['Psh']:.3f}  Q {terms['Q']:.3f} ({terms['q_mbps']:.1f} Mbit/s)  T {terms['T']:.3f} s  "
          f"r {r:+.4f}")
print("band and power fully shared over the interval (checked by the environment)")

## 8. Actor and critic

### Actor $\mu(S\mid\theta^\mu)$: the policy
$28 \to 128 \to 128$ (ReLU), then two heads:
* **bandwidth:** $L + 1 = 11$ logits $\to$ softmax $\to [b_0, b_1, \dots, b_L]$: the share kept for later videos, and
  the shares of the free band for the 10 groups;
* **power:** 1 logit $\to$ sigmoid $\to \rho$, the share of the free power.

The action is $a = [b_0, b_1..b_L, \rho]$: 12 numbers.

### Exploration
With probability $\varepsilon$ Gaussian noise $\mathcal N(0, 1)$ is added to **all 12 logits**, so the perturbed
action is still valid (shares sum to 1, $\rho \in (0,1)$).

### Critic $Q(S, a\mid\theta^Q)$
$[S, a]$ ($28 + 12 = 40$ numbers) $\to 128 \to 128$ (ReLU) $\to 1$.

**Starting weights.** Both actor heads start very small ($\pm 3\cdot 10^{-3}$), so the untrained actor keeps about
1/11 of the free band, splits the rest almost equally, and takes about half the free power. The cell prints this.

In [ ]:
S_DIM, A_DIM = 2 * L + 8, L + 2


class Actor(nn.Module):
    # mu(S | theta^mu): state -> logits -> a = [b_0 (keep), b_1..b_L, rho]
    def __init__(self, p=pcfg):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(S_DIM, p.hidden), nn.ReLU(), nn.Linear(p.hidden, p.hidden), nn.ReLU())
        self.to_b = nn.Linear(p.hidden, L + 1)           # bandwidth logits: keep, groups 1..L
        self.to_p = nn.Linear(p.hidden, 1)               # power logit
        for head in (self.to_b, self.to_p):
            nn.init.uniform_(head.weight, -3e-3, 3e-3); nn.init.uniform_(head.bias, -3e-3, 3e-3)

    def logits(self, s):
        h = self.body(s)
        return torch.cat([self.to_b(h), self.to_p(h)], -1)

    @staticmethod
    def to_action(lg):
        # b = softmax over the L + 1 bandwidth logits, rho = sigmoid of the power logit
        return torch.cat([torch.softmax(lg[..., :L + 1], -1), torch.sigmoid(lg[..., L + 1:])], -1)

    def forward(self, s):
        return self.to_action(self.logits(s))


class Critic(nn.Module):
    # Q(S, a | theta^Q)
    def __init__(self, p=pcfg):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(S_DIM + A_DIM, p.hidden), nn.ReLU(),
                                 nn.Linear(p.hidden, p.hidden), nn.ReLU(), nn.Linear(p.hidden, 1))

    def forward(self, s, a):
        return self.net(torch.cat([s, a], -1)).squeeze(-1)


def build_agent(gamma, p=pcfg):
    # actor, critic, their target copies and optimisers, for one discount gamma
    random.seed(p.seed); np.random.seed(p.seed); torch.manual_seed(p.seed)
    ag = SimpleNamespace(gamma=gamma, actor=Actor(p).to(DEVICE), critic=Critic(p).to(DEVICE),
                         actor_t=Actor(p).to(DEVICE), critic_t=Critic(p).to(DEVICE))
    ag.actor_t.load_state_dict(ag.actor.state_dict()); ag.critic_t.load_state_dict(ag.critic.state_dict())
    ag.opt_a = torch.optim.Adam(ag.actor.parameters(), lr=p.lr_actor)
    ag.opt_c = torch.optim.Adam(ag.critic.parameters(), lr=p.lr_critic)
    return ag


for name, net in (("actor  mu(S)", Actor()), ("critic Q(S,a)", Critic())):
    print(f"{name:14s} {n_par(net):7,} weights")
    for lname, m in net.named_modules():
        if isinstance(m, nn.Linear):
            print(f"    {lname:10s} {m.in_features:4d} -> {m.out_features:4d}   {n_par(m):6,}")
a0 = Actor().to(DEVICE)(torch.zeros(1, S_DIM, device=DEVICE))[0].detach().cpu().numpy()
print(f"untrained action: keep b_0 = {a0[0]:.3f}, group shares about {a0[1]:.3f} each, power rho = {a0[-1]:.3f}")

## 9. DDPG training

### One episode, step by step
A training episode draws its headsets, video choices, crowds and channel at random. Then, for every **step** (every
watched video in every interval):

1. **Observe** $S$. With probability $\varepsilon$ the action is the actor's output plus noise, otherwise the actor's
   output. The environment serves the video and returns the reward and the next state: the next video of the same
   interval, or the first video of the next interval. Store $\langle s, a, r, s', \text{done}\rangle$ in the
   **common** memory $\mathcal D$, with the action actually applied.
2. **If the memory holds more than 50 samples**, draw $M = 256$ (twice per step) and:
   * form the targets $y_j = r_j + \gamma(1-\text{done}_j)\,Q'(s'_j, \mu'(s'_j))$;
   * take one Adam step on the critic loss $\frac1M\sum_j (y_j - Q(s_j,a_j))^2$;
   * take one Adam step on the actor loss $-\frac1M\sum_j Q(s_j, \mu(s_j))$;
   * soft-update both targets with $\kappa = 0.01$.

After the episode, $\varepsilon \leftarrow 0.99\,\varepsilon$. `done` is set after the last step of interval 12.

**$\gamma$ now also discounts between videos of the same interval.** A step is one video, not one interval, so the
next video of the same interval is one step later.

### Tracking the critic's Q
Every update records the mean $Q(s_j, a_j)$ and target $y_j$ of its batch. Every 25 episodes the noise-free actor
plays the 40 fixed validation episodes. It records the critic's $Q(s, \mu(s))$ at every step and the discounted
reward that actually followed, $G_j = \sum_{i \ge 0}\gamma^i r_{j+i}$; the first step is $s_0$, $G_0$.

### Agents
1. **$\gamma$ sweep on 8K-20MHz:** $\gamma = 0.9, 0.95, 0.99$ with the same seed. The best is chosen by the
   **undiscounted** validation return over the last 10 validation runs.
2. **The chosen $\gamma$ on the other three scenarios** (8K-10MHz, 4K-20MHz, 4K-10MHz), one agent each. An agent
   only sees bandwidth and power as fractions, so it cannot tell that the band or the video changed. That is why
   each scenario has its own agent.

That makes 6 trainings. The cell prints a time estimate after 50 episodes. Set `RETRAIN = False` to load agents
saved by an earlier run; they are only used if they match this set-up (videos, headsets, dimensions, scenario,
$\gamma$).

In [ ]:
class Replay:
    # memory D of <s, a, r, s', done>, common to the steps of all videos
    def __init__(self, size, rng):
        self.s, self.s2 = np.zeros((size, S_DIM), np.float32), np.zeros((size, S_DIM), np.float32)
        self.a = np.zeros((size, A_DIM), np.float32)
        self.r, self.d = np.zeros(size, np.float32), np.zeros(size, np.float32)
        self.size, self.n, self.i, self.rng = size, 0, 0, rng

    def add(self, s, a, r, s2, done):
        self.s[self.i], self.a[self.i], self.r[self.i], self.s2[self.i], self.d[self.i] = s, a, r, s2, done
        self.i = (self.i + 1) % self.size
        self.n = min(self.n + 1, self.size)

    def sample(self, m):
        idx = self.rng.integers(0, self.n, m)
        return [torch.from_numpy(x[idx]).to(DEVICE) for x in (self.s, self.a, self.r, self.s2, self.d)]


def soft_update(target, source, kappa):
    with torch.no_grad():
        for t_, s_ in zip(target.parameters(), source.parameters()):
            t_.mul_(1 - kappa).add_(kappa * s_)


def ddpg_update(ag, batch, p=pcfg):
    s, a, r, s2, d = batch
    with torch.no_grad():                                                # y_j from the target networks
        y = r + ag.gamma * (1 - d) * ag.critic_t(s2, ag.actor_t(s2))
    q = ag.critic(s, a)
    critic_loss = torch.mean(torch.square(y - q))                        # mean of e_j^2
    ag.opt_c.zero_grad(); critic_loss.backward(); ag.opt_c.step()

    actor_loss = -ag.critic(s, ag.actor(s)).mean()                       # -Q(s_j, mu(s_j))
    ag.opt_a.zero_grad(); actor_loss.backward(); ag.opt_a.step()

    soft_update(ag.actor_t, ag.actor, p.soft_tau)                        # lines 15-16
    soft_update(ag.critic_t, ag.critic, p.soft_tau)
    return critic_loss.item(), actor_loss.item(), q.mean().item(), y.mean().item()


@torch.no_grad()
def act(ag, s, noise=0.0, rng=None):
    lg = ag.actor.logits(torch.from_numpy(s).to(DEVICE)[None])
    if noise:                                                            # N(0, noise^2) on all logits
        lg = lg + noise * torch.from_numpy(rng.standard_normal(lg.shape).astype(np.float32)).to(DEVICE)
    return Actor.to_action(lg)[0].cpu().numpy()


def run_episode(policy, split, seed, p=pcfg, rt=None, z_seed=None, keep_state=False, record=False):
    # play one episode with policy(env) -> action; returns the per-step terms (and the env if record)
    with torch.random.fork_rng(devices=[]):
        if z_seed is not None:
            torch.manual_seed(z_seed)                                    # same z_k draws for every policy
        env = StreamingEnv(split, seed, p, rt, record)
        env.reset(); done, rows = False, []
        while not done:
            s = env.state.copy()
            _, _, done, terms = env.step(policy(env))
            if keep_state:
                terms["s"] = s
            rows.append(terms)
    return (rows, env) if record else rows


def discounted_to_go(r, gamma):
    # G_j = r_j + gamma r_{j+1} + gamma^2 r_{j+2} + ...  (to the end of the episode)
    out, G_ = np.zeros(len(r)), 0.0
    for i in range(len(r) - 1, -1, -1):
        G_ = r[i] + gamma * G_
        out[i] = G_
    return out


def rank_shares(rows):
    # mean bandwidth and power share of each video rank over the steps of an episode (nan when nobody watched it)
    out = {}
    for v in range(V_N):
        rs = [t for t in rows if t["video"] == v]
        out[f"B_rank{v + 1}"] = np.mean([t["Bsh"] for t in rs]) if rs else np.nan
        out[f"P_rank{v + 1}"] = np.mean([t["Psh"] for t in rs]) if rs else np.nan
    return out


VAL_SEEDS = [10_000 + i for i in range(pcfg.n_val_episodes)]


@torch.no_grad()
def validate(ag, p, rt):
    # noise-free actor on the fixed validation episodes: undiscounted return, and the critic's Q against the realised G
    rets, q0, G0, q_all, G_all = [], [], [], [], []
    for sd in VAL_SEEDS:
        rows = run_episode(lambda env: act(ag, env.state), VAL, sd, p, rt, z_seed=sd, keep_state=True)
        r = np.array([t["r"] for t in rows])
        S = torch.from_numpy(np.stack([t["s"] for t in rows])).to(DEVICE)
        q = ag.critic(S, ag.actor(S)).cpu().numpy()
        G_ = discounted_to_go(r, ag.gamma)
        rets.append(r.sum()); q0.append(q[0]); G0.append(G_[0]); q_all.extend(q); G_all.extend(G_)
    return {"val_return": np.mean(rets), "val_q0": np.mean(q0), "val_G0": np.mean(G0),
            "val_q_all": np.mean(q_all), "val_G_all": np.mean(G_all)}


LOG_TERMS = ("QoE", "Q", "q_mbps", "T", "dQ", "Q_true", "Ub", "Ub_true", "Ud", "buf")
VAL_COLS = ("val_return", "val_q0", "val_G0", "val_q_all", "val_G_all")
N_TRAININGS = len(GAMMAS) + len(SCENARIOS) - 1


def train_agent(gamma, p=pcfg, rt=None, tag=""):
    rt = rt if rt is not None else rates(p)
    ag = build_agent(gamma, p)
    rng_t = np.random.default_rng(p.seed)
    memory = Replay(p.buffer, rng_t)
    eps, log, t0 = p.eps0, [], time.time()
    for ep in range(1, p.episodes + 1):
        env = StreamingEnv(TRAIN, seed=int(rng_t.integers(1 << 31)), p=p, rt=rt)
        s = env.reset()
        done, rows, stats = False, [], []
        while not done:
            # 1. observe s_t; with prob eps the action is actor o/p + noise, otherwise actor o/p
            noisy = rng_t.random() < eps
            a = act(ag, s, p.noise_std if noisy else 0.0, rng_t)
            #    reward from state and action, next state; store <s, a, r, s'> (the action actually applied)
            s2, r, done, terms = env.step(a)
            memory.add(s, env.applied_action, r, s2, float(done))
            rows.append(terms)
            # 2. enough samples in the buffer: update critic, actor and the targets
            if memory.n > p.min_buffer:
                for _ in range(p.updates_per_step):
                    stats.append(ddpg_update(ag, memory.sample(p.batch), p))
            s = s2
        eps *= p.eps_decay                                               # line 17

        st = np.array(stats) if stats else np.full((1, 4), np.nan)
        row = {"episode": ep, "eps": eps, "return": sum(t["r"] for t in rows), "steps": len(rows),
               **{k: np.mean([t[k] for t in rows]) for k in LOG_TERMS}, **rank_shares(rows),
               "critic_loss": st[:, 0].mean(), "actor_loss": st[:, 1].mean(),
               "q_batch": st[:, 2].mean(), "q_target": st[:, 3].mean(), **{c: np.nan for c in VAL_COLS}}
        if ep % p.eval_every == 0 or ep == p.episodes:
            row.update(validate(ag, p, rt))
        log.append(row)
        if ep == min(50, p.episodes):
            per = (time.time() - t0) / ep
            print(f"  [{tag}] {per:.2f} s per episode -> about {per * p.episodes / 60:.0f} min for this training, "
                  f"{per * p.episodes * N_TRAININGS / 3600:.1f} h for all {N_TRAININGS}")
        if ep % 500 == 0 or ep == 1:
            last, lg = pd.DataFrame(log[-100:]), pd.DataFrame(log)
            vr = lg.val_return.dropna()
            print(f"[{tag}] episode {ep:5d}  eps {eps:.3f}  return {last['return'].mean():7.4f}  "
                  f"Q {last.Q.mean():.3f}  T {last['T'].mean():.3f}s  |dQ| {last.dQ.mean():.3f}  "
                  f"critic {last.critic_loss.mean():.5f}  Qcritic {last.q_batch.mean():6.3f}  "
                  f"val return {vr.iloc[-1] if len(vr) else float('nan'):7.4f}  ({time.time() - t0:.0f}s)")
    return ag, pd.DataFrame(log)


RETRAIN = True        # False: load the agents saved by an earlier run of this cell (when they match this set-up)
gtag = lambda g: f"g{g:g}"
agent_files = lambda tag: [os.path.join(OUT_DIR, f) for f in
                           (f"ddpg_actor_{tag}.pt", f"ddpg_critic_{tag}.pt", f"ddpg_episode_log_{tag}.csv")]
META = {"L": L, "S_DIM": S_DIM, "A_DIM": A_DIM, "videos": list(pcfg.videos), "n_users": pcfg.n_users,
        "zipf_s": pcfg.zipf_s, "q_mode": pcfg.q_mode}


def train_or_load(tag, gamma, scen):
    fa, fc, fl = agent_files(tag)
    p_, rt_ = SCEN_CFG[scen], SCEN_RT[scen]
    if not RETRAIN and all(os.path.exists(f) for f in (fa, fc, fl)):
        ck = torch.load(fa, map_location=DEVICE, weights_only=False)
        if all(ck.get(k) == val for k, val in META.items()) and ck.get("scenario") == scen and ck.get("gamma") == gamma:
            ag = build_agent(gamma, p_)
            ag.actor.load_state_dict(ck["actor"])
            ag.critic.load_state_dict(torch.load(fc, map_location=DEVICE, weights_only=False)["critic"])
            print(f"{tag}: loaded {os.path.basename(fa)} (set RETRAIN = True to train it again)")
            return ag, pd.read_csv(fl)
        print(f"{tag}: {os.path.basename(fa)} is from another set-up, training again")
    print(f"--- training '{tag}': scenario {scen}, gamma {gamma:g} ---")
    ag, log = train_agent(gamma, p_, rt_, tag)
    torch.save({"actor": ag.actor.state_dict(), "gamma": gamma, "scenario": scen, "pcfg": asdict(p_), **META}, fa)
    torch.save({"critic": ag.critic.state_dict(), "gamma": gamma, "scenario": scen}, fc)
    log.to_csv(fl, index=False)
    return ag, log


AGENTS, LOGS, t_all = {}, {}, time.time()
for g in GAMMAS:                                                   # 1. the gamma sweep on the base scenario
    AGENTS[(BASE_SCEN, g)], LOGS[(BASE_SCEN, g)] = train_or_load(gtag(g), g, BASE_SCEN)

# ---- choose gamma: the highest undiscounted validation return over the last 10 validation runs ----
GSEL = pd.DataFrame({g: {"validation return (last 10 runs)": LOGS[(BASE_SCEN, g)].val_return.dropna().tail(10).mean(),
                         "critic Q(s_0, mu(s_0))": LOGS[(BASE_SCEN, g)].val_q0.dropna().tail(10).mean(),
                         "realised G_0": LOGS[(BASE_SCEN, g)].val_G0.dropna().tail(10).mean()} for g in GAMMAS}).T
GSEL.index.name = "gamma"
GAMMA_BEST = float(GSEL["validation return (last 10 runs)"].idxmax())
print(GSEL.round(4).to_string())
print(f"chosen on {BASE_SCEN}: gamma = {GAMMA_BEST:g}")

for scen in SCENARIOS:                                             # 2. the chosen gamma on the other scenarios
    if scen != BASE_SCEN:
        AGENTS[(scen, GAMMA_BEST)], LOGS[(scen, GAMMA_BEST)] = train_or_load(scen, GAMMA_BEST, scen)
CHOSEN = {scen: AGENTS[(scen, GAMMA_BEST)] for scen in SCENARIOS}
DLOG = LOGS[(BASE_SCEN, GAMMA_BEST)]
ddpg_policy = lambda env: act(CHOSEN[BASE_SCEN], env.state)
print(f"\n{len(LOGS)} agents in {time.time() - t_all:.0f}s on {DEVICE.type}, {pcfg.episodes} episodes each")
print(pd.DataFrame({scen: {"validation return (last 10 runs)": LOGS[(scen, GAMMA_BEST)].val_return.dropna().tail(10).mean()}
                    for scen in SCENARIOS}).T.round(4).to_string())

## 10. Training graphs

| panel | what it shows | what to look for |
|---|---|---|
| **return** (8K-20MHz) | 50-episode moving average of the training return per $\gamma$, and the validation return (dots) | it should rise and level off |
| **critic loss**, **actor loss** | per $\gamma$ | the critic loss falls; the actor loss falls as Q rises |
| **Seer's QoE terms** | $\lambda Q$, $-\mu T$, $-\xi\lvert\Delta Q\rvert$ and their sum, per headset, chosen $\gamma$ | how quality is traded against freezes and quality changes |
| **Q, predicted vs true FoV** | $Q$ on $\hat F$ (the reward) and on the true $G_{u,k}$ | the gap is the cost of prediction error |
| **playout** | mean freeze $T$ and mean buffer | freezes should fall |
| **bandwidth / power share per video** | mean $B_v/B_\text{tot}$ and $P_v/P_\text{tot}$ of each Zipf rank | does the agent give popular videos more? An equal split is 1/V ≈ 0.2, a split by audience follows $p_v$ |
| **return per scenario** | the four agents with the chosen $\gamma$ | the levels differ between scenarios; compare each with its baselines in section 11 |

In [ ]:
ma = lambda x, w=50: pd.Series(x).rolling(w, min_periods=1).mean()
GCOL = dict(zip(GAMMAS, ["#1f77b4", "#ff7f0e", "#2ca02c"]))
RCOL = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2"]
_, LAM, MU_R, XI = seer_weights(pcfg, RT)
fig, axes = plt.subplots(3, 3, figsize=(16, 11))
for g in GAMMAS:
    D = LOGS[(BASE_SCEN, g)]
    v = D.dropna(subset=["val_return"])
    axes[0, 0].plot(D.episode, ma(D["return"]), color=GCOL[g], label=rf"$\gamma$ = {g:g}" + (" (chosen)" if g == GAMMA_BEST else ""))
    axes[0, 0].plot(v.episode, v.val_return, "o", ms=2.5, alpha=.6, color=GCOL[g])
    axes[0, 1].plot(D.episode, ma(D.critic_loss), color=GCOL[g], lw=.9, label=rf"$\gamma$ = {g:g}")
    axes[0, 2].plot(D.episode, ma(D.actor_loss), color=GCOL[g], lw=.9, label=rf"$\gamma$ = {g:g}")
axes[0, 0].set_ylabel(r"return $\sum r$"); axes[0, 0].set_title(f"{BASE_SCEN}: return (lines: training MA 50, dots: validation)")
axes[0, 1].set_yscale("log"); axes[0, 1].set_title(r"Critic loss $\mathcal{L}(\theta^Q)$ (MA 50)")
axes[0, 2].set_title(r"Actor loss $-Q(s, \mu(s))$ (MA 50)")

ax = axes[1, 0]
ax.plot(DLOG.episode, ma(LAM * DLOG.Q), color="#1f77b4", label=r"$\lambda Q$")
ax.plot(DLOG.episode, ma(-MU_R * DLOG["T"]), color="#d62728", label=r"$-\mu T$ (rebuffering)")
ax.plot(DLOG.episode, ma(-XI * DLOG.dQ), color="#ff7f0e", label=r"$-\xi\,|\Delta Q|$ (smoothness)")
ax.plot(DLOG.episode, ma(DLOG.QoE), color="k", lw=1, label="QoE per headset")
ax.set_title(rf"Seer's QoE terms per headset (MA 50), $\gamma$ = {GAMMA_BEST:g}")

ax = axes[1, 1]
ax.plot(DLOG.episode, ma(DLOG.Q), color="#1f77b4", label=r"$Q$ on predicted FoV $\hat F_k$ (reward)")
ax.plot(DLOG.episode, ma(DLOG.Q_true), color="#2ca02c", label=r"$Q$ on true FoV $G_{u,k}$ (reported)")
ax.set_title("Quality Q: predicted vs true FoV")

ax = axes[1, 2]
ax.plot(DLOG.episode, ma(DLOG["T"]), color="#d62728", label="mean freeze T per headset and step (s)")
ax2 = ax.twinx(); ax2.grid(False)
ax2.plot(DLOG.episode, ma(DLOG.buf), color="#7f7f7f", label="mean playout buffer (s, right)")
ax.set_title(rf"Playout (MA 50), $\gamma$ = {GAMMA_BEST:g}")
ax.legend(handles=ax.get_lines() + ax2.get_lines(), fontsize=8)

for i, (key, lbl) in enumerate((("B", "bandwidth"), ("P", "power"))):
    ax = axes[2, i]
    for r_ in range(V_N):
        ax.plot(DLOG.episode, ma(DLOG[f"{key}_rank{r_ + 1}"]), color=RCOL[r_],
                label=f"rank {r_ + 1}: {pcfg.videos[r_]} (p = {P_ZIPF[r_]:.2f})")
    ax.set_title(f"{lbl} share of each video (MA 50), $\\gamma$ = {GAMMA_BEST:g}"); ax.set_ylim(0, 1)
ax = axes[2, 2]
for i, scen in enumerate(SCENARIOS):
    D = LOGS[(scen, GAMMA_BEST)]
    ax.plot(D.episode, ma(D["return"]), color=RCOL[i], label=scen)
ax.set_title(rf"Training return per scenario (MA 50), $\gamma$ = {GAMMA_BEST:g}")
for a_ in axes.ravel():
    a_.set_xlabel("episode")
    if a_ is not axes[1, 2]:
        a_.legend(fontsize=7)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "ddpg_training.png"), dpi=130)
plt.show()

### The critic's Q against episodes

**Left:** the critic's mean $Q(s_j, a_j)$ on the replay batches (solid) and the mean target $y_j$ (dotted), per
$\gamma$, on 8K-20MHz. **Right:** on the validation episodes, the critic's value of the actor's own first action,
$Q(s_0, \mu(s_0))$ (solid), against the discounted reward that actually followed, $G_0$ (dashed). The two
right-hand curves should meet; a solid line above its dashed line means overestimation. The table lists the last 10
validation runs of every agent, including the other three scenarios.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4.4))
for g in GAMMAS:
    D = LOGS[(BASE_SCEN, g)]
    v = D.dropna(subset=["val_q0"])
    ax1.plot(D.episode, ma(D.q_batch), color=GCOL[g], label=rf"$\gamma$ = {g:g}: $Q(s_j, a_j)$")
    ax1.plot(D.episode, ma(D.q_target), color=GCOL[g], ls=":", lw=1.2, label=rf"$\gamma$ = {g:g}: target $y_j$")
    ax2.plot(v.episode, v.val_q0, "-", color=GCOL[g], label=rf"$\gamma$ = {g:g}: critic $Q(s_0, \mu(s_0))$")
    ax2.plot(v.episode, v.val_G0, "--", color=GCOL[g], label=rf"$\gamma$ = {g:g}: realised $G_0$")
ax1.set_title(f"{BASE_SCEN}: critic Q on replay batches (MA 50)"); ax1.set_ylabel("Q")
ax2.set_title("Validation: critic Q of the first step vs the discounted reward that followed")
for a_ in (ax1, ax2):
    a_.set_xlabel("episode"); a_.legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "ddpg_q_values.png"), dpi=130)
plt.show()

QTAB = pd.DataFrame({(scen, g): {"Q(s_0, mu(s_0))": D.val_q0.dropna().tail(10).mean(),
                                 "G_0": D.val_G0.dropna().tail(10).mean(),
                                 "Q - G, first step": (D.val_q0 - D.val_G0).dropna().tail(10).mean(),
                                 "Q - G, all steps": (D.val_q_all - D.val_G_all).dropna().tail(10).mean()}
                     for (scen, g), D in LOGS.items()}).T
QTAB.index.names = ["scenario", "gamma"]
print("last 10 validation runs:")
print(QTAB.round(4).to_string())

## 11. Test: DDPG against eight rules that need no learning, in every scenario

All policies play the **same 50 test episodes** in each scenario: the same headset positions, walks, shadowing,
video choices and draws of $z$. So every difference comes from the allocation alone. The crowds are the test crowds
`U_test[v]`.

**Rules.** Each rule combines a split over the groups with a split over the videos:

| group rule ($b_1..b_L$, as a split of the video's band) | video rule (the video's fraction $f$ of the free band and power) |
|---|---|
| **attention**: $\propto T_l w_l$ | **equal split**: $f = 1/(\text{watched videos left})$, so every watched video gets $B_\text{tot}/V_\text{active}$ |
| **equal per tile**: $\propto T_l$ | **split by audience**: $f = N_v / \sum_{\text{videos left}} N_{v'}$, so video $v$ gets $\propto N_v$ |
| **demand**: $\propto T_l S_l$ | |
| **random**: $\mathrm{Dirichlet}(\mathbf 1_L)$ | |

Every rule gives a video the same fraction of the free **power** as of the free **band** (flat PSD across the
cell).

**Rows per scenario:**
* "DDPG": the scenario's own agent with the chosen $\gamma$;
* on 8K-20MHz, the other two $\gamma$;
* on the other scenarios, the 8K-20MHz agent as a robustness check;
* the 8 rules.

**Columns:**
* `return`: the sum of the step rewards, i.e. the mean over the 50 headsets of their QoE summed over the 7
  intervals, ± its standard error;
* per step and headset: `QoE`, `Q`, `q_mbps` (Mbit/s), `T` (freeze, s), `dQ`, `Q_true`, and the reported `Ub` and
  `Ud`;
* `buf`, the playout buffer (s).

Then the mean bandwidth and power share each video rank receives.

**What to expect.** The policy matters most where bandwidth binds: 8K, and especially 8K at 10 MHz (section 7's
demand table). At 4K and 20 MHz most sensible rules deliver nearly every target layer, and DDPG has little room to
win.

In [ ]:
def share(x):
    return x / x.sum()


GROUP_RULES = {
    "attention":      lambda env: share(env.T * env.w),
    "equal per tile": lambda env: share(env.T),
    "demand":         lambda env: share(env.T * env.rt.S_CUM),
}
rng_rand = np.random.default_rng(pcfg.seed + 1)
GROUP_RULES["random"] = lambda env: rng_rand.dirichlet(np.ones(L))
VIDEO_RULES = {
    "equal split":       lambda env: 1.0 / env.n_left,                  # B_tot / V_active for every video
    "split by audience": lambda env: env.N_v / env.N_left_sum,          # proportional to N_v
}


def baseline(group_rule, video_rule):
    # the video takes the fraction f of the free band and of the free power (flat PSD), split over groups by the rule
    def pol(env):
        g, f = group_rule(env), video_rule(env)
        return np.concatenate([[1.0 - f], f * g, [f]])
    return pol


BASELINES = {f"{gn} | {vn}": baseline(gr, vr) for gn, gr in GROUP_RULES.items() for vn, vr in VIDEO_RULES.items()}
TEST_SEEDS = [20_000 + i for i in range(pcfg.n_test_seeds)]


def policies_for(scen):
    pols = {"DDPG": lambda env, ag=CHOSEN[scen]: act(ag, env.state)}
    if scen == BASE_SCEN:
        pols.update({f"DDPG gamma={g:g}": (lambda env, ag=AGENTS[(BASE_SCEN, g)]: act(ag, env.state))
                     for g in GAMMAS if g != GAMMA_BEST})
    else:
        pols[f"DDPG ({BASE_SCEN} agent)"] = lambda env: act(CHOSEN[BASE_SCEN], env.state)   # robustness
    return {**pols, **BASELINES}


rows = []
for scen in SCENARIOS:
    for name, pol in policies_for(scen).items():
        for sd in TEST_SEEDS:
            for t in run_episode(pol, TEST, sd, SCEN_CFG[scen], SCEN_RT[scen], z_seed=sd):
                rows.append({"scenario": scen, "policy": name, "seed": sd, **t})
TRES = pd.DataFrame(rows)

ep_ret = TRES.groupby(["scenario", "policy", "seed"]).r.sum()
gr_ = ep_ret.groupby(["scenario", "policy"])
TSUM = pd.DataFrame({"return": gr_.mean(), "return s.e.": gr_.std() / np.sqrt(pcfg.n_test_seeds),
                     **{f"mean {k}": TRES.groupby(["scenario", "policy"])[k].mean()
                        for k in ("QoE", "Q", "q_mbps", "T", "dQ", "Q_true", "Ub", "Ud", "buf")}})
TSUM.to_csv(os.path.join(OUT_DIR, "ddpg_test_summary.csv"))
print(f"test crowds {dict((v, U_test[v]) for v in pcfg.videos)}\n{pcfg.n_test_seeds} test episodes per policy and "
      f"scenario; gamma = {GAMMA_BEST:g} (chosen on {BASE_SCEN})")
for scen in SCENARIOS:
    T_ = TSUM.loc[scen].sort_values("return", ascending=False)
    print(f"\n===== {scen} =====")
    print(T_.round(4).to_string())
    best_rule = T_.loc[list(BASELINES)]["return"].idxmax()
    dd = ep_ret.loc[(scen, "DDPG")]
    diff = dd - ep_ret.loc[(scen, best_rule)]
    print(f"DDPG minus the best baseline ({best_rule}): {diff.mean():+.4f} per episode "
          f"(s.e. {diff.std() / np.sqrt(len(diff)):.4f}, DDPG better in {(diff > 0).sum()} of {len(diff)} episodes)")

SHARES = (TRES[TRES.policy.isin(["DDPG", "attention | equal split", "attention | split by audience"])]
          .groupby(["scenario", "policy", "video"])[["Bsh", "Psh"]].mean().unstack("video"))
SHARES.columns = [f"{k} rank {v + 1}" for k, v in SHARES.columns]
print("\nmean bandwidth (Bsh) and power (Psh) share per video rank:")
print(SHARES.round(3).to_string())

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, scen in zip(axes.ravel(), SCENARIOS):
    T_ = TSUM.loc[scen].sort_values("return")
    ax.barh(T_.index, T_["return"], xerr=T_["return s.e."],
            color=["#d62728" if p_.startswith("DDPG") else "#1f77b4" for p_ in T_.index])
    ax.set_title(f"{scen}: mean test return (Seer's QoE)"); ax.tick_params(axis="y", labelsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "ddpg_test_policies.png"), dpi=130)
plt.show()

## 12. One interval up close

The first test episode of 8K-20MHz, first interval ($k = 6$), with the chosen DDPG agent. One row per watched video:
* the predicted frame $\hat F_k$;
* the bandwidth $B(i,j)$ its tiles get;
* the mean number of layers its headsets receive;
* the delay utility per tile.

Bottom: the bandwidth and power share of each video, for DDPG and for "attention | equal split".

What to look for: does DDPG give the popular videos more, and does it move power where it is worth most?

In [ ]:
def first_interval(policy, scen=BASE_SCEN):
    # the first test episode (seed and z_k draws), its first interval: one entry per watched video
    _, env_ = run_episode(policy, TEST, TEST_SEEDS[0], SCEN_CFG[scen], SCEN_RT[scen], z_seed=TEST_SEEDS[0], record=True)
    return [h for h in env_.history if h["t"] == 0]


H_dd = first_interval(ddpg_policy)
H_eq = first_interval(BASELINES["attention | equal split"])
nv = len(H_dd)
fig = plt.figure(figsize=(16, 2.4 * nv + 3.2))
gs = fig.add_gridspec(nv + 1, 4)
for r_, h in enumerate(H_dd):
    panels = ((h["F"], r"$\hat F_k$", "inferno", (0, 1)), (h["B"] / 1e3, r"$B(i,j)$ (kHz)", "magma", None),
              (h["n_mean"], r"mean layers $n_u(i,j)$", "cividis", (0, L)),
              (h["Ud_tile"], "delay utility per tile", "viridis", (0, 1)))
    for c_, (img, lbl, cm, lim) in enumerate(panels):
        ax = fig.add_subplot(gs[r_, c_])
        im = ax.imshow(img, cmap=cm, aspect="auto", **({"vmin": lim[0], "vmax": lim[1]} if lim else {}))
        fig.colorbar(im, ax=ax)
        ax.set_title(f"{pcfg.videos[h['video']]} (N_v = {h['N_v']}): " + lbl, fontsize=8)
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)

names = [pcfg.videos[h["video"]] for h in H_dd]
xs, wd = np.arange(nv), 0.38
for c_, (key, lbl) in enumerate((("Bsh", "bandwidth share"), ("Psh", "power share"))):
    ax = fig.add_subplot(gs[nv, 2 * c_:2 * c_ + 2])
    ax.bar(xs - wd / 2, [h[key] for h in H_dd], wd, label=f"DDPG ($\\gamma$ = {GAMMA_BEST:g})", color="#d62728")
    ax.bar(xs + wd / 2, [h[key] for h in H_eq], wd, label="attention | equal split", color="#1f77b4")
    ax.set_xticks(xs); ax.set_xticklabels(names); ax.set_title(f"{lbl} of each video, k = {KS[0]}"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "ddpg_one_interval.png"), dpi=130)
plt.show()
for lbl, H in (("DDPG", H_dd), ("attention | equal split", H_eq)):
    print(f"{lbl:24s} " + "  ".join(f"{pcfg.videos[h['video']]}: Q {h['Q']:.3f} T {h['T']:.3f}s" for h in H))

## 13. Files, and points to confirm

**Files written to `outputs_predictor_vae_combined/`**

| file | contents |
|---|---|
| `ddpg_actor.pt`, `ddpg_critic.pt`, `ddpg_episode_log.csv` | the chosen 8K-20MHz agent: actor (with $\gamma$, `PCfg`, videos, headsets, dims, encoder and predictor file names), critic, training log |
| `ddpg_actor_g0.9.pt`, `ddpg_critic_g0.9.pt`, `ddpg_episode_log_g0.9.csv`, … | every $\gamma$ of the sweep on 8K-20MHz |
| `ddpg_actor_8K-10MHz.pt`, … `_4K-20MHz`, `_4K-10MHz` (with critic and log) | the chosen $\gamma$ on the other scenarios |
| `ddpg_test_summary.csv` | the test table of section 11, all scenarios |
| `ddpg_masks.png`, `ddpg_channel.png`, `ddpg_training.png`, `ddpg_q_values.png`, `ddpg_test_policies.png`, `ddpg_one_interval.png` | the figures |

**Serving.** While streaming, only the actor is needed. In every interval it is queried once per watched video in
rank order, with the free bandwidth and power updated after each query.

**Points still to be confirmed**

1. **$Q$ in Seer's reward (open):** option A (normalised, `q_mode = "normalised"`, the default) or option B (Mbit/s
   with $\mu = Q_\text{max}$, `"mbps"`). Same optimal policy; A trains on rewards of order 1.
2. **Seer's reward, mapped onto SVC tiles:** $q$ is the $\hat F$-weighted, whole-frame-equivalent delivered bitrate,
   and $T$ is the viewer's total freeze in the segment. Our segments are 5 s; Seer's weights were set for its own
   segment length.
3. **Bitrates:** anchored at 40 Mbit/s for all layers of a 4K frame (YouTube's upload guidance). Delivery ladders
   in the literature are lower. 4K matches the dataset; **8K is a hypothetical re-encoding** of the same content,
   with head motion assumed independent of resolution.
4. **Zipf:** $s = 0.8$, with ranks in the order of `videos` (no popularity data). Choices are fixed for an episode.
5. **Power:** each video's power is spread evenly over its band (flat PSD); the total is 23 dBm.
6. **The last watched video of an interval takes everything left** (its action is overridden).
7. **A viewer who receives nothing freezes for the whole segment** ($T = 5$ s). This closes a loophole in which a
   fully starved video scored 0 instead of a penalty. Individual starved tiles still do not freeze playback.
8. **Two added state entries:** $N_v/50$ and the number of videos still to come.
9. **Behaviours:** 10 per video. With more than 10 headsets on a video, behaviours repeat; headsets that share one
   still have their own channels.
10. **The common encoder** must be trained on the pooled frames of all 5 videos; the cell warns if it does not
   list them.
11. **Test crowd and the front end:** if the predictors were trained on all 50 viewers, the test crowds are unseen
    by the policy only.
12. **Channel, playout queue, equal-width quantiles, continuous bandwidth shares:** as in the single-video version
    (sections 6 and 7).
13. **Training time:** about 35 steps per episode and 6 trainings, roughly 30× the single-video version.

[1] L. Liang, H. Ye, and G. Y. Li, "Spectrum sharing in vehicular networks based on multi-agent reinforcement
learning," *IEEE J. Sel. Areas Commun.*, vol. 37, no. 10, pp. 2282–2292, Oct. 2019.

In [ ]:
ag = CHOSEN[BASE_SCEN]
torch.save({"actor": ag.actor.state_dict(), "gamma": GAMMA_BEST, "scenario": BASE_SCEN, "pcfg": asdict(pcfg), **META,
            "encoder_file": os.path.basename(ENC_FILE),
            "predictor_files": {v: os.path.basename(f) for v, f in PRED_FILE.items()}},
           os.path.join(OUT_DIR, "ddpg_actor.pt"))
torch.save({"critic": ag.critic.state_dict(), "gamma": GAMMA_BEST, "scenario": BASE_SCEN},
           os.path.join(OUT_DIR, "ddpg_critic.pt"))
DLOG.to_csv(os.path.join(OUT_DIR, "ddpg_episode_log.csv"), index=False)
saved = ["ddpg_actor.pt", "ddpg_critic.pt", "ddpg_episode_log.csv", "ddpg_test_summary.csv",
         *(os.path.basename(f) for g in GAMMAS for f in agent_files(gtag(g))),
         *(os.path.basename(f) for scen in SCENARIOS if scen != BASE_SCEN for f in agent_files(scen))]
for f in saved:
    print("saved", os.path.join(OUT_DIR, f))